# NB12 - Native Mindray File Final Closure and Pipeline Utility Summary

## Purpose

This notebook consolidates the native-file exploration work across DopplerLab.

It reproduces the strongest surviving native-file findings, closes unsupported native extraction paths, and summarizes what native files can usefully contribute to the DopplerLab pipeline.

## Scope

In scope:

- native recording manifest,
- native page timebase,
- native 20 Hz counter,
- `hp_activity` timing/QC signal,
- native QC classes,
- audio validation as external reference,
- closure of unsupported extraction paths,
- final native-file utility matrix.

Out of scope:

- pipeline refactor,
- new clinical validation,
- production modularization,
- pressure estimation,
- calibrated native velocity extraction,
- native spectrogram recovery,
- native PSV, EDV, RI, PI, or VTI extraction.

## Interpretation boundary

All native-derived physiological outputs in this notebook are experimental and non-clinical.

This notebook does not claim calibrated velocity, pressure, native Doppler indices, native spectrogram recovery, PHYSIO recovery, or diagnostic interpretation.


## Import and Setup

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as sg
from scipy.ndimage import uniform_filter1d

In [3]:
PAGE_SIZE_BYTES = 1296
PAGE_RATE_HZ = 500.0
HEADER_BYTES = 16
RECORD_BYTES = 8
N_RECORDS = 160
N_LIVE_RECORDS = 16
N_HP_RECORDS = 14
ENV_RATE_HZ = 100.0
COUNTER_OFFSET = 1248
COUNTER_WIDTH = 2
SAVE_OUTPUTS = False

In [4]:
def find_project_root(start_path=None):
    """This function finds the DopplerLab project root from the current notebook location."""
    if start_path is None:
        start_path = Path.cwd()

    start_path = Path(start_path).resolve()

    for candidate in [start_path, *start_path.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "ultrasound_recordings").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find DopplerLab project root. "
        "Run this notebook from inside the DopplerLab repository."
    )

In [5]:
PROJECT_ROOT = find_project_root()

NATIVE_ROOT = PROJECT_ROOT / "ultrasound_recordings"
AUDIO_EXPORT_ROOT = PROJECT_ROOT / "audio_exports"
SCREENING_EXPORT_ROOT = PROJECT_ROOT / "screening_exports"
FEATURE_EXPORT_ROOT = PROJECT_ROOT / "feature_exports"

REPORT_ROOT = PROJECT_ROOT / "reports" / "nb12_native_file_closure"
FIGURE_ROOT = PROJECT_ROOT / "figures" / "nb12_native_file_closure"
DOC_ROOT = PROJECT_ROOT / "docs" / "nb12_native_file_closure"

path_check = pd.DataFrame(
    [
        {"name": "PROJECT_ROOT", "path": PROJECT_ROOT, "exists": PROJECT_ROOT.exists()},
        {"name": "NATIVE_ROOT", "path": NATIVE_ROOT, "exists": NATIVE_ROOT.exists()},
        {"name": "AUDIO_EXPORT_ROOT", "path": AUDIO_EXPORT_ROOT, "exists": AUDIO_EXPORT_ROOT.exists()},
        {"name": "SCREENING_EXPORT_ROOT", "path": SCREENING_EXPORT_ROOT, "exists": SCREENING_EXPORT_ROOT.exists()},
        {"name": "FEATURE_EXPORT_ROOT", "path": FEATURE_EXPORT_ROOT, "exists": FEATURE_EXPORT_ROOT.exists()},
        {"name": "REPORT_ROOT", "path": REPORT_ROOT, "exists": REPORT_ROOT.exists()},
        {"name": "FIGURE_ROOT", "path": FIGURE_ROOT, "exists": FIGURE_ROOT.exists()},
        {"name": "DOC_ROOT", "path": DOC_ROOT, "exists": DOC_ROOT.exists()},
    ]
)

native_recording_dirs = sorted(
    path for path in NATIVE_ROOT.iterdir()
    if path.is_dir()
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Native recording folders found: {len(native_recording_dirs)}")
print("Output folders are defined but not created because SAVE_OUTPUTS is False.")

display(path_check)
display(pd.DataFrame({"recording_id": [path.name for path in native_recording_dirs]}))

Project root: D:\code\DopplerLab
Native recording folders found: 2
Output folders are defined but not created because SAVE_OUTPUTS is False.


,name,path,exists
0,PROJECT_ROOT,D:\code\DopplerLab,True
1,NATIVE_ROOT,D:\code\DopplerLab\ultrasound_recordings,True
2,AUDIO_EXPORT_ROOT,D:\code\DopplerLab\audio_exports,True
3,SCREENING_EXPORT_ROOT,D:\code\DopplerLab\screening_exports,True
4,FEATURE_EXPORT_ROOT,D:\code\DopplerLab\feature_exports,True
5,REPORT_ROOT,D:\code\DopplerLab\reports\nb12_native_file_cl...,False
6,FIGURE_ROOT,D:\code\DopplerLab\figures\nb12_native_file_cl...,False
7,DOC_ROOT,D:\code\DopplerLab\docs\nb12_native_file_closure,False


,recording_id
0,batch_2026_06_13
1,batch_2026_06_13_native


## Native recording manifest


In [6]:
def file_size_or_nan(path):
    """This function returns a file size in bytes when a path exists."""
    path = Path(path)
    if path.exists():
        return path.stat().st_size
    return np.nan


def find_first_existing(recording_dir, relative_candidates):
    """This function returns the first existing file from a list of relative candidate paths."""
    for relative_path in relative_candidates:
        candidate = recording_dir / relative_path
        if candidate.exists():
            return candidate
    return None


def describe_native_pw_location(pw_path):
    """This function identifies the recording folder and native-file folder for a PW file."""
    pw_path = Path(pw_path)

    native_file_dir = pw_path.parent

    if native_file_dir.name.lower() == "native":
        recording_dir = native_file_dir.parent
        recording_id = recording_dir.name
        batch_folder = recording_dir.parent.name
    else:
        recording_dir = native_file_dir
        recording_id = recording_dir.name
        batch_folder = recording_dir.parent.name

    return recording_id, batch_folder, recording_dir, native_file_dir


def build_native_recording_manifest(native_root):
    """This function discovers native recording folders and summarizes expected native files."""
    native_root = Path(native_root)

    pw_files = sorted(native_root.rglob("PW_CinePartition0.bin"))

    rows = []
    for pw_path in pw_files:
        recording_id, batch_folder, recording_dir, native_file_dir = describe_native_pw_location(pw_path)

        bc_path = native_file_dir / "BC_CinePartition1.bin"
        dcm_region_path = find_first_existing(
            native_file_dir,
            [
                Path("DcmRegionPara.txt"),
                Path("DCMRegionPara.txt"),
                Path("DcmRegionPara.xml"),
            ],
        )
        virtual_machine_path = native_file_dir / "VirtualMachine.bin"

        candidate_avi_files = sorted(PROJECT_ROOT.rglob(f"*{recording_id}*.avi"))
        candidate_audio_tables = sorted(AUDIO_EXPORT_ROOT.rglob(f"*{recording_id}*.csv"))
        candidate_screening_tables = sorted(SCREENING_EXPORT_ROOT.rglob(f"*{recording_id}*.csv"))

        rows.append(
            {
                "recording_id": recording_id,
                "batch_folder": batch_folder,
                "recording_dir": recording_dir,
                "native_file_dir": native_file_dir,
                "has_pw_cinepartition0": pw_path.exists(),
                "pw_cinepartition0_path": pw_path,
                "pw_cinepartition0_size_bytes": file_size_or_nan(pw_path),
                "has_bc_cinepartition1": bc_path.exists(),
                "bc_cinepartition1_path": bc_path,
                "bc_cinepartition1_size_bytes": file_size_or_nan(bc_path),
                "has_dcm_region_para": dcm_region_path is not None,
                "dcm_region_para_path": dcm_region_path,
                "has_virtual_machine_bin": virtual_machine_path.exists(),
                "virtual_machine_path": virtual_machine_path,
                "virtual_machine_size_bytes": file_size_or_nan(virtual_machine_path),
                "candidate_avi_count": len(candidate_avi_files),
                "candidate_audio_table_count": len(candidate_audio_tables),
                "candidate_screening_table_count": len(candidate_screening_tables),
            }
        )

    manifest = pd.DataFrame(rows)

    if not manifest.empty:
        manifest = manifest.sort_values(["batch_folder", "recording_id"]).reset_index(drop=True)

    return manifest

In [7]:
native_manifest = build_native_recording_manifest(NATIVE_ROOT)

print(f"Native recordings with PW_CinePartition0.bin found: {len(native_manifest)}")

if native_manifest.empty:
    print("No PW_CinePartition0.bin files were found under NATIVE_ROOT.")
else:
    display(
        native_manifest[
            [
                "recording_id",
                "batch_folder",
                "has_pw_cinepartition0",
                "pw_cinepartition0_size_bytes",
                "has_bc_cinepartition1",
                "bc_cinepartition1_size_bytes",
                "has_dcm_region_para",
                "has_virtual_machine_bin",
                "candidate_avi_count",
                "candidate_audio_table_count",
                "candidate_screening_table_count",
            ]
        ]
    )

    display(
        native_manifest[
            [
                "recording_id",
                "recording_dir",
                "native_file_dir",
                "dcm_region_para_path",
            ]
        ]
    )

Native recordings with PW_CinePartition0.bin found: 10


,recording_id,batch_folder,has_pw_cinepartition0,pw_cinepartition0_size_bytes,has_bc_cinepartition1,bc_cinepartition1_size_bytes,has_dcm_region_para,has_virtual_machine_bin,candidate_avi_count,candidate_audio_table_count,candidate_screening_table_count
0,202606130411060002SMP,batch_2026_06_13_native,True,38381040,True,174368,True,True,1,0,0
1,202606130413540003SMP,batch_2026_06_13_native,True,30444336,True,174368,True,True,1,0,0
2,202606130417060004SMP,batch_2026_06_13_native,True,27516672,True,174368,True,True,1,0,0
3,202606130420260005SMP,batch_2026_06_13_native,True,29502144,True,174368,True,True,1,0,0
4,202606130422440006SMP,batch_2026_06_13_native,True,33211296,True,174368,True,True,1,0,0
5,202606130426500007SMP,batch_2026_06_13_native,True,23776416,True,174368,True,True,1,0,0
6,202606130430380008SMP,batch_2026_06_13_native,True,28753056,True,174368,True,True,1,0,0
7,202606130433280009SMP,batch_2026_06_13_native,True,20557152,True,174368,True,True,1,0,0
8,202606130434130010SMP,batch_2026_06_13_native,True,21923136,True,174368,True,True,1,0,0
9,202606130437480012SMP,batch_2026_06_13_native,True,29176848,True,232088,True,True,1,0,0


,recording_id,recording_dir,native_file_dir,dcm_region_para_path
0,202606130411060002SMP,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...
1,202606130413540003SMP,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...
2,202606130417060004SMP,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...
3,202606130420260005SMP,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...
4,202606130422440006SMP,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...
5,202606130426500007SMP,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...
6,202606130430380008SMP,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...
7,202606130433280009SMP,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...
8,202606130434130010SMP,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...
9,202606130437480012SMP,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...,D:\code\DopplerLab\ultrasound_recordings\batch...


## PW page timebase reproduction

### What this tests

This section tests whether `PW_CinePartition0.bin` supports fixed-page decomposition across all native recordings.

### Why this matters

The fixed-page structure is the foundation for the native timebase, the 20 Hz counter check, and the later `hp_activity` timing/QC signal. A clean page decomposition supports treating the file as a structured native PW stream.



In [8]:
def find_candidate_avi_path(recording_id):
    """This function finds a candidate linked AVI file by recording ID."""
    matches = sorted(PROJECT_ROOT.rglob(f"*{recording_id}*.avi"))
    if len(matches) == 1:
        return matches[0]
    return None


def read_avi_duration_seconds(avi_path):
    """This function reads AVI duration from frame count and FPS when OpenCV is available."""
    if avi_path is None:
        return np.nan, np.nan, np.nan

    try:
        import cv2
    except ImportError:
        return np.nan, np.nan, np.nan

    capture = cv2.VideoCapture(str(avi_path))

    if not capture.isOpened():
        capture.release()
        return np.nan, np.nan, np.nan

    frame_count = capture.get(cv2.CAP_PROP_FRAME_COUNT)
    fps = capture.get(cv2.CAP_PROP_FPS)
    capture.release()

    if fps <= 0 or frame_count <= 0:
        return np.nan, frame_count, fps

    duration_s = frame_count / fps
    return duration_s, frame_count, fps


def build_pw_timebase_qc(manifest):
    """This function checks fixed-page PW decomposition and estimates native duration."""
    rows = []

    for row in manifest.itertuples(index=False):
        pw_path = Path(row.pw_cinepartition0_path)
        file_size_bytes = pw_path.stat().st_size

        n_pages = file_size_bytes // PAGE_SIZE_BYTES
        remainder_bytes = file_size_bytes % PAGE_SIZE_BYTES
        native_duration_s = n_pages / PAGE_RATE_HZ

        avi_path = find_candidate_avi_path(row.recording_id)
        avi_duration_s, avi_frame_count, avi_fps = read_avi_duration_seconds(avi_path)

        if np.isfinite(avi_duration_s):
            native_minus_avi_s = native_duration_s - avi_duration_s
        else:
            native_minus_avi_s = np.nan

        rows.append(
            {
                "recording_id": row.recording_id,
                "pw_size_bytes": file_size_bytes,
                "page_size_bytes": PAGE_SIZE_BYTES,
                "page_remainder_bytes": remainder_bytes,
                "page_decomposition_ok": remainder_bytes == 0,
                "n_pages": n_pages,
                "page_rate_hz": PAGE_RATE_HZ,
                "native_duration_s": native_duration_s,
                "candidate_avi_path": avi_path,
                "avi_duration_s": avi_duration_s,
                "avi_frame_count": avi_frame_count,
                "avi_fps": avi_fps,
                "native_minus_avi_s": native_minus_avi_s,
                "abs_native_minus_avi_s": abs(native_minus_avi_s)
                if np.isfinite(native_minus_avi_s)
                else np.nan,
            }
        )

    qc = pd.DataFrame(rows)
    qc = qc.sort_values("recording_id").reset_index(drop=True)
    return qc

In [9]:
pw_timebase_qc = build_pw_timebase_qc(native_manifest)

print("PW fixed-page decomposition summary")
print(f"Recordings checked: {len(pw_timebase_qc)}")
print(f"Page decomposition OK: {int(pw_timebase_qc['page_decomposition_ok'].sum())} / {len(pw_timebase_qc)}")
print(f"OpenCV AVI durations available: {int(pw_timebase_qc['avi_duration_s'].notna().sum())} / {len(pw_timebase_qc)}")

display(
    pw_timebase_qc[
        [
            "recording_id",
            "pw_size_bytes",
            "page_remainder_bytes",
            "page_decomposition_ok",
            "n_pages",
            "native_duration_s",
            "avi_duration_s",
            "avi_fps",
            "native_minus_avi_s",
            "abs_native_minus_avi_s",
        ]
    ]
)

display(
    pw_timebase_qc[
        [
            "recording_id",
            "candidate_avi_path",
        ]
    ]
)

PW fixed-page decomposition summary
Recordings checked: 10
Page decomposition OK: 10 / 10
OpenCV AVI durations available: 10 / 10


,recording_id,pw_size_bytes,page_remainder_bytes,page_decomposition_ok,n_pages,native_duration_s,avi_duration_s,avi_fps,native_minus_avi_s,abs_native_minus_avi_s
0,202606130411060002SMP,38381040,0,True,29615,59.230,59.266667,30.0,-0.036667,0.036667
1,202606130413540003SMP,30444336,0,True,23491,46.982,47.033333,30.0,-0.051333,0.051333
2,202606130417060004SMP,27516672,0,True,21232,42.464,42.500000,30.0,-0.036000,0.036000
3,202606130420260005SMP,29502144,0,True,22764,45.528,45.566667,30.0,-0.038667,0.038667
4,202606130422440006SMP,33211296,0,True,25626,51.252,51.300000,30.0,-0.048000,0.048000
5,202606130426500007SMP,23776416,0,True,18346,36.692,36.733333,30.0,-0.041333,0.041333
6,202606130430380008SMP,28753056,0,True,22186,44.372,44.433333,30.0,-0.061333,0.061333
7,202606130433280009SMP,20557152,0,True,15862,31.724,31.766667,30.0,-0.042667,0.042667
8,202606130434130010SMP,21923136,0,True,16916,33.832,33.866667,30.0,-0.034667,0.034667
9,202606130437480012SMP,29176848,0,True,22513,45.026,45.066667,30.0,-0.040667,0.040667


,recording_id,candidate_avi_path
0,202606130411060002SMP,D:\code\DopplerLab\ultrasound_recordings\batch...
1,202606130413540003SMP,D:\code\DopplerLab\ultrasound_recordings\batch...
2,202606130417060004SMP,D:\code\DopplerLab\ultrasound_recordings\batch...
3,202606130420260005SMP,D:\code\DopplerLab\ultrasound_recordings\batch...
4,202606130422440006SMP,D:\code\DopplerLab\ultrasound_recordings\batch...
5,202606130426500007SMP,D:\code\DopplerLab\ultrasound_recordings\batch...
6,202606130430380008SMP,D:\code\DopplerLab\ultrasound_recordings\batch...
7,202606130433280009SMP,D:\code\DopplerLab\ultrasound_recordings\batch...
8,202606130434130010SMP,D:\code\DopplerLab\ultrasound_recordings\batch...
9,202606130437480012SMP,D:\code\DopplerLab\ultrasound_recordings\batch...


## 20 Hz page-group counter reproduction

### What this tests

This section tests whether bytes `1248..1249` in each `1296` byte PW page behave like a page-group counter.

### Why this matters

A stable page-group counter can support native acquisition-integrity checks. If the counter increments every `25` pages at a `500 Hz` page rate, the implied counter rate is `20 Hz`.

In [10]:
def read_pw_pages_uint8(pw_path):
    """This function reads a PW file as fixed-size uint8 pages."""
    pw_path = Path(pw_path)
    raw = np.fromfile(pw_path, dtype=np.uint8)

    if raw.size % PAGE_SIZE_BYTES != 0:
        raise ValueError(
            f"PW file does not divide into {PAGE_SIZE_BYTES} byte pages: {pw_path}"
        )

    return raw.reshape((-1, PAGE_SIZE_BYTES))


def decode_page_counter_u16(pages):
    """This function decodes the candidate little-endian uint16 page-group counter."""
    counter_bytes = pages[:, COUNTER_OFFSET : COUNTER_OFFSET + COUNTER_WIDTH]
    counter = counter_bytes[:, 0].astype(np.uint16) + (
        counter_bytes[:, 1].astype(np.uint16) << 8
    )
    return counter


def summarize_counter_runs(counter):
    """This function summarizes consecutive page runs with the same counter value."""
    counter = np.asarray(counter)

    if counter.size == 0:
        return pd.DataFrame(columns=["counter_value", "start_page", "end_page", "run_length_pages"])

    change_idx = np.flatnonzero(np.diff(counter) != 0) + 1
    starts = np.r_[0, change_idx]
    ends = np.r_[change_idx - 1, counter.size - 1]

    runs = pd.DataFrame(
        {
            "counter_value": counter[starts].astype(int),
            "start_page": starts.astype(int),
            "end_page": ends.astype(int),
            "run_length_pages": (ends - starts + 1).astype(int),
        }
    )

    return runs


def summarize_pw_counter_qc(manifest):
    """This function checks the candidate 20 Hz page-group counter for each recording."""
    rows = []

    for row in manifest.itertuples(index=False):
        pages = read_pw_pages_uint8(row.pw_cinepartition0_path)
        counter = decode_page_counter_u16(pages)
        runs = summarize_counter_runs(counter)

        counter_diffs = np.diff(counter.astype(np.int64))
        run_lengths = runs["run_length_pages"].to_numpy()

        interior_run_lengths = run_lengths
        if len(run_lengths) > 2:
            interior_run_lengths = run_lengths[1:-1]

        expected_group_pages = 25
        median_run_length = float(np.median(interior_run_lengths)) if len(interior_run_lengths) else np.nan
        implied_counter_rate_hz = PAGE_RATE_HZ / median_run_length if median_run_length > 0 else np.nan

        nonzero_diffs = counter_diffs[counter_diffs != 0]
        positive_step_count = int(np.sum(nonzero_diffs > 0))
        negative_step_count = int(np.sum(nonzero_diffs < 0))

        expected_interior_fraction = (
            float(np.mean(interior_run_lengths == expected_group_pages))
            if len(interior_run_lengths)
            else np.nan
        )

        rows.append(
            {
                "recording_id": row.recording_id,
                "n_pages": pages.shape[0],
                "n_counter_runs": len(runs),
                "first_counter": int(counter[0]),
                "last_counter": int(counter[-1]),
                "min_counter": int(counter.min()),
                "max_counter": int(counter.max()),
                "median_run_length_pages": median_run_length,
                "expected_25_page_interior_fraction": expected_interior_fraction,
                "implied_counter_rate_hz": implied_counter_rate_hz,
                "positive_counter_step_count": positive_step_count,
                "negative_counter_step_count": negative_step_count,
                "has_negative_counter_step": negative_step_count > 0,
                "largest_abs_counter_step": int(np.max(np.abs(nonzero_diffs))) if len(nonzero_diffs) else 0,
                "first_run_length_pages": int(run_lengths[0]) if len(run_lengths) else np.nan,
                "last_run_length_pages": int(run_lengths[-1]) if len(run_lengths) else np.nan,
            }
        )

    qc = pd.DataFrame(rows)
    qc = qc.sort_values("recording_id").reset_index(drop=True)
    return qc

In [11]:
pw_counter_qc = summarize_pw_counter_qc(native_manifest)

print("PW candidate counter summary")
print(f"Recordings checked: {len(pw_counter_qc)}")
print(
    "Recordings with median 25-page counter runs: "
    f"{int((pw_counter_qc['median_run_length_pages'] == 25).sum())} / {len(pw_counter_qc)}"
)
print(
    "Recordings with no negative counter step: "
    f"{int((~pw_counter_qc['has_negative_counter_step']).sum())} / {len(pw_counter_qc)}"
)

display(pw_counter_qc)

PW candidate counter summary
Recordings checked: 10
Recordings with median 25-page counter runs: 10 / 10
Recordings with no negative counter step: 10 / 10


,recording_id,n_pages,n_counter_runs,first_counter,last_counter,min_counter,max_counter,median_run_length_pages,expected_25_page_interior_fraction,implied_counter_rate_hz,positive_counter_step_count,negative_counter_step_count,has_negative_counter_step,largest_abs_counter_step,first_run_length_pages,last_run_length_pages
0,202606130411060002SMP,29615,1180,6124,35714,6124,35714,25.0,0.902377,20.0,1179,0,False,26,25,25
1,202606130413540003SMP,23491,936,1556,25022,1556,25022,25.0,0.902570,20.0,935,0,False,26,25,25
2,202606130417060004SMP,21232,846,1004,22211,1004,22211,25.0,0.902844,20.0,845,0,False,26,25,25
3,202606130420260005SMP,22764,907,2258,24997,2258,24997,25.0,0.902762,20.0,906,0,False,26,26,25
4,202606130422440006SMP,25626,1021,727,26327,727,26327,25.0,0.902846,20.0,1020,0,False,26,26,26
5,202606130426500007SMP,18346,731,1330,19651,1330,19651,25.0,0.902606,20.0,730,0,False,26,25,25
6,202606130430380008SMP,22186,884,7906,30067,7906,30067,25.0,0.902494,20.0,883,0,False,26,25,25
7,202606130433280009SMP,15862,632,1280,17116,1280,17116,25.0,0.903175,20.0,631,0,False,26,25,26
8,202606130434130010SMP,16916,674,803,17694,803,17694,25.0,0.901786,20.0,673,0,False,26,25,25
9,202606130437480012SMP,22513,897,953,23441,953,23441,25.0,0.901676,20.0,896,0,False,26,25,25


### 20 Hz page-group counter interpretation

Result classification:

This section supports bytes `1248..1249` as a native page-group timing and acquisition-integrity field.

Evidence-supported observations:

- All ten recordings have a median counter run length of `25` pages.
- At a `500 Hz` page rate, `25` pages imply a `20 Hz` page-group update rate.
- No recording shows a negative counter step.
- The candidate counter field is monotone across all ten recordings.
- The largest observed nonzero page-to-page step is `26`, so this field behaves more like a native timing/sample counter than a simple `+1` group index.
- Approximately `90%` of interior runs are exactly `25` pages, with small deviations consistent with counter/timing quantization or native bookkeeping.

What this supports:

- Bytes `1248..1249` can be used as a native acquisition-integrity and timing/QC sidecar.
- The field can help detect page-order problems, resets, or discontinuities in future recordings.
- The `500 Hz` page-rate and `20 Hz` counter-rate assumptions are mutually consistent.

What this does not support:

- This field is not a physiological waveform.
- This field does not encode a Doppler velocity envelope.
- This field does not recover a native spectrogram, pressure trace, PHYSIO trace, or clinical measurement.


## Reproduce `hp_activity`

### What this tests

This section tests whether records `0..13`, bytes `[4:8]`, can be read from raw PW pages as float32-like dynamic values and transformed into the canonical `hp_activity` timing/QC signal.

### Why this matters

`hp_activity` is the strongest surviving native-derived signal candidate from the PW stream. Reproducing it directly from raw bytes supports treating it as an experimental arbitrary-unit timing/QC sidecar.


In [12]:
def extract_live_record_float_values(pages, n_records=N_HP_RECORDS):
    """This function extracts bytes [4:8] from live PW records as little-endian float32 values."""
    if pages.ndim != 2 or pages.shape[1] != PAGE_SIZE_BYTES:
        raise ValueError("Expected pages with shape (n_pages, PAGE_SIZE_BYTES).")

    record_start = HEADER_BYTES
    record_end = HEADER_BYTES + (n_records * RECORD_BYTES)

    live_record_bytes = pages[:, record_start:record_end]
    live_record_bytes = live_record_bytes.reshape((pages.shape[0], n_records, RECORD_BYTES))

    value_bytes = np.ascontiguousarray(live_record_bytes[:, :, 4:8])
    values = value_bytes.view("<f4").reshape((pages.shape[0], n_records))

    return values


def compute_hp_activity(live_values, baseline_size_pages=51):
    """This function computes an arbitrary-unit high-pass activity signal from live values."""
    baseline = uniform_filter1d(
        live_values,
        size=baseline_size_pages,
        axis=0,
        mode="nearest",
    )
    hp_activity = np.abs(live_values - baseline).mean(axis=1)
    return hp_activity


def summarize_hp_activity(manifest):
    """This function reproduces hp_activity from raw PW files and summarizes basic signal checks."""
    rows = []
    signals = {}

    for row in manifest.itertuples(index=False):
        pages = read_pw_pages_uint8(row.pw_cinepartition0_path)
        live_values = extract_live_record_float_values(pages, n_records=N_HP_RECORDS)
        hp_activity = compute_hp_activity(live_values, baseline_size_pages=51)
        hp_activity_25 = compute_hp_activity(live_values, baseline_size_pages=25)

        time_s = np.arange(len(hp_activity)) / PAGE_RATE_HZ

        finite_fraction = float(np.mean(np.isfinite(live_values)))
        hp_finite_fraction = float(np.mean(np.isfinite(hp_activity)))

        rows.append(
            {
                "recording_id": row.recording_id,
                "n_pages": pages.shape[0],
                "duration_s": pages.shape[0] / PAGE_RATE_HZ,
                "live_value_shape": live_values.shape,
                "live_value_finite_fraction": finite_fraction,
                "live_value_min": float(np.nanmin(live_values)),
                "live_value_max": float(np.nanmax(live_values)),
                "hp_activity_finite_fraction": hp_finite_fraction,
                "hp_activity_min": float(np.nanmin(hp_activity)),
                "hp_activity_median": float(np.nanmedian(hp_activity)),
                "hp_activity_max": float(np.nanmax(hp_activity)),
                "hp_activity_p95": float(np.nanpercentile(hp_activity, 95)),
                "hp_activity_25_median": float(np.nanmedian(hp_activity_25)),
                "hp_activity_25_p95": float(np.nanpercentile(hp_activity_25, 95)),
            }
        )

        signals[row.recording_id] = {
            "time_s": time_s,
            "live_values": live_values,
            "hp_activity": hp_activity,
            "hp_activity_25": hp_activity_25,
        }

    summary = pd.DataFrame(rows).sort_values("recording_id").reset_index(drop=True)
    return summary, signals

In [13]:
hp_activity_summary, hp_activity_signals = summarize_hp_activity(native_manifest)

print("hp_activity reproduction summary")
print(f"Recordings checked: {len(hp_activity_summary)}")
print(
    "Recordings with all finite live values: "
    f"{int((hp_activity_summary['live_value_finite_fraction'] == 1.0).sum())} / {len(hp_activity_summary)}"
)
print(
    "Recordings with all finite hp_activity: "
    f"{int((hp_activity_summary['hp_activity_finite_fraction'] == 1.0).sum())} / {len(hp_activity_summary)}"
)

display(hp_activity_summary)

hp_activity reproduction summary
Recordings checked: 10
Recordings with all finite live values: 10 / 10
Recordings with all finite hp_activity: 10 / 10


,recording_id,n_pages,duration_s,live_value_shape,live_value_finite_fraction,live_value_min,live_value_max,hp_activity_finite_fraction,hp_activity_min,hp_activity_median,hp_activity_max,hp_activity_p95,hp_activity_25_median,hp_activity_25_p95
0,202606130411060002SMP,29615,59.230,"(29615, 14)",1.0,-4.981574,4.868809,1.0,0.000053,0.085996,5.612390,1.964565,0.027942,1.336426
1,202606130413540003SMP,23491,46.982,"(23491, 14)",1.0,-3.840626,3.819148,1.0,0.001475,0.215519,5.827619,1.834071,0.050120,1.527684
2,202606130417060004SMP,21232,42.464,"(21232, 14)",1.0,-3.658003,3.770095,1.0,0.000895,0.267421,4.144230,2.077255,0.052960,1.409343
3,202606130420260005SMP,22764,45.528,"(22764, 14)",1.0,-5.116026,5.062938,1.0,0.001215,1.865366,6.374360,4.261484,0.979017,3.648120
4,202606130422440006SMP,25626,51.252,"(25626, 14)",1.0,-3.937287,3.910350,1.0,0.003685,1.026501,5.257068,2.607745,0.443175,2.295203
5,202606130426500007SMP,18346,36.692,"(18346, 14)",1.0,-5.034291,5.033323,1.0,0.000134,1.711449,6.980158,4.166113,1.062670,3.782523
6,202606130430380008SMP,22186,44.372,"(22186, 14)",1.0,-4.592056,4.649367,1.0,0.001131,0.150451,5.215333,1.601511,0.055435,1.355186
7,202606130433280009SMP,15862,31.724,"(15862, 14)",1.0,-5.099367,5.074097,1.0,0.001618,1.400995,6.942845,3.929592,0.653463,2.697366
8,202606130434130010SMP,16916,33.832,"(16916, 14)",1.0,-4.668557,4.699215,1.0,0.001667,1.135760,6.324412,3.455659,0.378450,2.433033
9,202606130437480012SMP,22513,45.026,"(22513, 14)",1.0,-5.224118,5.105661,1.0,0.000730,0.354840,5.487936,1.995965,0.045865,1.440063


### `hp_activity` reproduction interpretation

Result classification:

This section supports direct raw-byte reproduction of the canonical native `hp_activity` signal across the current ten-recording batch.

Evidence-supported observations:

- All ten recordings produced finite live values from records `0..13`, bytes `[4:8]`.
- Each live-value array has shape `(n_pages, 14)`.
- All ten recordings produced finite `hp_activity` values.
- The reproduced signal amplitude differs substantially across recordings.
- The shorter `hp_activity_25` variant produces lower median and percentile values, as expected from a shorter local baseline.

What this supports:

- Records `0..13`, bytes `[4:8]`, contain reproducible dynamic float32-like native values.
- `hp_activity` can be computed directly from raw `PW_CinePartition0.bin` pages without helper scripts.
- `hp_activity` remains a viable candidate arbitrary-unit native timing/QC sidecar for further testing.

What this does not support:

- This result does not prove that `hp_activity` is cardiac-specific.
- This result does not validate native beat timing.
- This result does not decode calibrated native velocity, native Doppler indices, pressure, or a clinical measurement.
- Larger `hp_activity` amplitude does not automatically imply better signal quality.


## Native HR and within-recording stability from `hp_activity`

### What this tests

This section tests whether `hp_activity` contains a stable dominant cardiac-band rhythm within each recording.

### Why this matters

A native-derived timing/QC sidecar should show stable within-recording rate behavior in usable recordings. Instability, harmonic behavior, or inconsistent window estimates would support reject or audio-only classification.


In [14]:
def resample_signal_to_rate(signal_values, source_rate_hz, target_rate_hz):
    """This function resamples a one-dimensional signal to a lower target rate."""
    signal_values = np.asarray(signal_values, dtype=float)
    duration_s = len(signal_values) / source_rate_hz
    n_target = int(np.floor(duration_s * target_rate_hz))

    source_time_s = np.arange(len(signal_values)) / source_rate_hz
    target_time_s = np.arange(n_target) / target_rate_hz

    resampled = np.interp(target_time_s, source_time_s, signal_values)
    return target_time_s, resampled


def estimate_hr_from_psd(signal_values, rate_hz, min_bpm=45, max_bpm=180):
    """This function estimates dominant HR from the PSD peak in a plausible cardiac band."""
    signal_values = np.asarray(signal_values, dtype=float)

    if len(signal_values) < rate_hz * 4:
        return np.nan, np.nan

    centered = signal_values - np.nanmedian(signal_values)
    centered = np.nan_to_num(centered, nan=0.0)

    nperseg = min(len(centered), int(rate_hz * 8))
    if nperseg < int(rate_hz * 4):
        return np.nan, np.nan

    freqs_hz, power = sg.welch(
        centered,
        fs=rate_hz,
        nperseg=nperseg,
        noverlap=nperseg // 2,
        detrend="constant",
    )

    band_mask = (freqs_hz >= min_bpm / 60.0) & (freqs_hz <= max_bpm / 60.0)

    if not np.any(band_mask):
        return np.nan, np.nan

    band_freqs = freqs_hz[band_mask]
    band_power = power[band_mask]

    peak_index = int(np.argmax(band_power))
    peak_bpm = float(band_freqs[peak_index] * 60.0)

    total_power = float(np.sum(band_power))
    peak_power_fraction = float(band_power[peak_index] / total_power) if total_power > 0 else np.nan

    return peak_bpm, peak_power_fraction


def estimate_window_hr_series(signal_values, rate_hz, window_s=10.0, step_s=5.0):
    """This function estimates HR in sliding windows."""
    signal_values = np.asarray(signal_values, dtype=float)

    window_n = int(round(window_s * rate_hz))
    step_n = int(round(step_s * rate_hz))

    rows = []
    for start in range(0, max(0, len(signal_values) - window_n + 1), step_n):
        stop = start + window_n
        window = signal_values[start:stop]

        hr_bpm, peak_fraction = estimate_hr_from_psd(window, rate_hz)

        rows.append(
            {
                "start_s": start / rate_hz,
                "stop_s": stop / rate_hz,
                "center_s": (start + stop) / (2 * rate_hz),
                "window_hr_bpm": hr_bpm,
                "window_peak_power_fraction": peak_fraction,
            }
        )

    return pd.DataFrame(rows)


def summarize_native_hr_from_hp_activity(signals):
    """This function summarizes native HR and window stability from hp_activity."""
    rows = []
    window_tables = {}

    for recording_id, signal_dict in signals.items():
        time_s, hp_100 = resample_signal_to_rate(
            signal_dict["hp_activity"],
            source_rate_hz=PAGE_RATE_HZ,
            target_rate_hz=ENV_RATE_HZ,
        )

        hr_bpm, peak_fraction = estimate_hr_from_psd(hp_100, ENV_RATE_HZ)

        window_hr = estimate_window_hr_series(
            hp_100,
            rate_hz=ENV_RATE_HZ,
            window_s=10.0,
            step_s=5.0,
        )

        finite_window_hr = window_hr["window_hr_bpm"].dropna()

        if len(finite_window_hr) > 0:
            window_hr_median = float(np.median(finite_window_hr))
            window_hr_iqr = float(
                np.percentile(finite_window_hr, 75) - np.percentile(finite_window_hr, 25)
            )
            window_hr_min = float(np.min(finite_window_hr))
            window_hr_max = float(np.max(finite_window_hr))
        else:
            window_hr_median = np.nan
            window_hr_iqr = np.nan
            window_hr_min = np.nan
            window_hr_max = np.nan

        rows.append(
            {
                "recording_id": recording_id,
                "duration_s": float(time_s[-1]) if len(time_s) else np.nan,
                "global_native_hr_bpm": hr_bpm,
                "global_peak_power_fraction": peak_fraction,
                "n_hr_windows": int(len(finite_window_hr)),
                "window_hr_median_bpm": window_hr_median,
                "window_hr_iqr_bpm": window_hr_iqr,
                "window_hr_min_bpm": window_hr_min,
                "window_hr_max_bpm": window_hr_max,
            }
        )

        window_tables[recording_id] = window_hr

    summary = pd.DataFrame(rows).sort_values("recording_id").reset_index(drop=True)
    return summary, window_tables

In [15]:
native_hr_summary, native_hr_windows = summarize_native_hr_from_hp_activity(hp_activity_signals)

print("Native HR summary from hp_activity")
print(f"Recordings checked: {len(native_hr_summary)}")
print(
    "Recordings with window HR IQR < 8 bpm: "
    f"{int((native_hr_summary['window_hr_iqr_bpm'] < 8).sum())} / {len(native_hr_summary)}"
)

display(native_hr_summary)

Native HR summary from hp_activity
Recordings checked: 10
Recordings with window HR IQR < 8 bpm: 5 / 10


,recording_id,duration_s,global_native_hr_bpm,global_peak_power_fraction,n_hr_windows,window_hr_median_bpm,window_hr_iqr_bpm,window_hr_min_bpm,window_hr_max_bpm
0,202606130411060002SMP,59.22,60.0,0.175574,10,63.75,61.875,60.0,180.0
1,202606130413540003SMP,46.97,52.5,0.237613,8,52.50,13.125,52.5,112.5
2,202606130417060004SMP,42.45,112.5,0.393882,7,112.50,0.000,112.5,180.0
3,202606130420260005SMP,45.51,60.0,0.252463,8,67.50,7.500,60.0,67.5
4,202606130422440006SMP,51.24,60.0,0.271998,9,60.00,0.000,60.0,120.0
5,202606130426500007SMP,36.68,97.5,0.130005,6,112.50,20.625,67.5,135.0
6,202606130430380008SMP,44.36,45.0,0.105586,7,105.00,71.250,45.0,180.0
7,202606130433280009SMP,31.71,105.0,0.458371,5,105.00,0.000,105.0,112.5
8,202606130434130010SMP,33.82,105.0,0.405844,5,105.00,7.500,105.0,112.5
9,202606130437480012SMP,45.01,67.5,0.168268,8,63.75,9.375,45.0,127.5


### Native HR stability interpretation

Result classification:

This section supports the presence of cardiac-band rhythm-like structure in `hp_activity`, but it does not support using simple window HR stability as a final native-specific classifier.

Evidence-supported observations:

- All ten recordings produced global cardiac-band HR estimates from `hp_activity`.
- Five of ten recordings had window HR IQR below `8 bpm`.
- Several recordings show unstable or harmonic-sensitive window estimates.
- `202606130433280009SMP` and `202606130434130010SMP` appear stable by this simple window-IQR check, even though they require stricter native-specific QC before adoption.
- `202606130411060002SMP` and `202606130413540003SMP` show instability under this simple PSD-window method, which is consistent with harmonic sensitivity or window-level ambiguity rather than a clean pass/fail conclusion.

What this supports:

- `hp_activity` can contain dominant cardiac-band timing structure.
- A native HR estimate can be produced from the arbitrary-unit native signal.
- Window-level stability is a useful QC feature.

What this does not support:

- Window HR IQR alone does not classify native-specific recordings.
- This result does not validate native beat times.
- This result does not prove that stable native HR is physiologically specific.
- This result does not validate calibrated velocity, native Doppler indices, pressure, or clinical measurements.


## Native HR robustness QC

### What this tests

This section tests whether native HR estimates from `hp_activity` remain stable across signal subsets and high-pass baseline choices.

### Why this matters

A useful native timing/QC sidecar should not depend on one fragile filtering choice or one part of the recording. Subset disagreement and filter sensitivity provide stricter evidence than simple global HR or window HR IQR alone.



In [16]:
def estimate_native_hr_for_activity(activity_values, source_rate_hz=PAGE_RATE_HZ, env_rate_hz=ENV_RATE_HZ):
    """This function estimates HR from a native activity signal after resampling."""
    _, env_signal = resample_signal_to_rate(
        activity_values,
        source_rate_hz=source_rate_hz,
        target_rate_hz=env_rate_hz,
    )
    hr_bpm, peak_fraction = estimate_hr_from_psd(env_signal, env_rate_hz)
    return hr_bpm, peak_fraction


def estimate_subset_hr_values(activity_values, source_rate_hz=PAGE_RATE_HZ, env_rate_hz=ENV_RATE_HZ):
    """This function estimates HR in first half, second half, even pages, and odd pages."""
    activity_values = np.asarray(activity_values, dtype=float)

    midpoint = len(activity_values) // 2

    subsets = {
        "first_half": activity_values[:midpoint],
        "second_half": activity_values[midpoint:],
        "even_pages": activity_values[::2],
        "odd_pages": activity_values[1::2],
    }

    subset_rows = []
    for subset_name, subset_values in subsets.items():
        subset_source_rate_hz = source_rate_hz

        if subset_name in ["even_pages", "odd_pages"]:
            subset_source_rate_hz = source_rate_hz / 2.0

        hr_bpm, peak_fraction = estimate_native_hr_for_activity(
            subset_values,
            source_rate_hz=subset_source_rate_hz,
            env_rate_hz=env_rate_hz,
        )

        subset_rows.append(
            {
                "subset": subset_name,
                "hr_bpm": hr_bpm,
                "peak_power_fraction": peak_fraction,
            }
        )

    return pd.DataFrame(subset_rows)


def summarize_native_hr_robustness(signals, window_summary):
    """This function summarizes subset agreement and filter sensitivity for native HR."""
    rows = []
    subset_tables = {}

    for recording_id, signal_dict in signals.items():
        hp_activity = signal_dict["hp_activity"]
        hp_activity_25 = signal_dict["hp_activity_25"]

        hr_51, peak_51 = estimate_native_hr_for_activity(hp_activity)
        hr_25, peak_25 = estimate_native_hr_for_activity(hp_activity_25)

        subset_table = estimate_subset_hr_values(hp_activity)
        finite_subset_hr = subset_table["hr_bpm"].dropna()

        if len(finite_subset_hr) > 0:
            subset_disagreement_bpm = float(finite_subset_hr.max() - finite_subset_hr.min())
            subset_hr_values = ", ".join(f"{value:.1f}" for value in finite_subset_hr)
        else:
            subset_disagreement_bpm = np.nan
            subset_hr_values = ""

        filter_sensitivity_bpm = abs(hr_51 - hr_25) if np.isfinite(hr_51) and np.isfinite(hr_25) else np.nan

        matching_window = window_summary.loc[window_summary["recording_id"] == recording_id].iloc[0]

        rows.append(
            {
                "recording_id": recording_id,
                "global_hr_hp51_bpm": hr_51,
                "global_peak_fraction_hp51": peak_51,
                "global_hr_hp25_bpm": hr_25,
                "global_peak_fraction_hp25": peak_25,
                "filter_sensitivity_bpm": filter_sensitivity_bpm,
                "subset_disagreement_bpm": subset_disagreement_bpm,
                "subset_hr_values_bpm": subset_hr_values,
                "window_hr_iqr_bpm": float(matching_window["window_hr_iqr_bpm"]),
                "passes_window_iqr_lt8": bool(matching_window["window_hr_iqr_bpm"] < 8),
                "passes_subset_disagreement_lt6": bool(subset_disagreement_bpm < 6),
                "passes_filter_sensitivity_lt8": bool(filter_sensitivity_bpm < 8),
            }
        )

        subset_tables[recording_id] = subset_table

    robustness = pd.DataFrame(rows).sort_values("recording_id").reset_index(drop=True)

    robustness["passes_native_robustness_qc"] = (
        robustness["passes_window_iqr_lt8"]
        & robustness["passes_subset_disagreement_lt6"]
        & robustness["passes_filter_sensitivity_lt8"]
    )

    return robustness, subset_tables

In [17]:
native_hr_robustness, native_hr_subset_tables = summarize_native_hr_robustness(
    hp_activity_signals,
    native_hr_summary,
)

print("Native HR robustness summary")
print(f"Recordings checked: {len(native_hr_robustness)}")
print(
    "Recordings passing window + subset + filter robustness QC: "
    f"{int(native_hr_robustness['passes_native_robustness_qc'].sum())} / {len(native_hr_robustness)}"
)

display(native_hr_robustness)

Native HR robustness summary
Recordings checked: 10
Recordings passing window + subset + filter robustness QC: 3 / 10


,recording_id,global_hr_hp51_bpm,global_peak_fraction_hp51,global_hr_hp25_bpm,global_peak_fraction_hp25,filter_sensitivity_bpm,subset_disagreement_bpm,subset_hr_values_bpm,window_hr_iqr_bpm,passes_window_iqr_lt8,passes_subset_disagreement_lt6,passes_filter_sensitivity_lt8,passes_native_robustness_qc
0,202606130411060002SMP,60.0,0.175574,60.0,0.159183,0.0,67.5,"60.0, 127.5, 60.0, 60.0",61.875,False,False,True,False
1,202606130413540003SMP,52.5,0.237613,52.5,0.210047,0.0,0.0,"52.5, 52.5, 52.5, 52.5",13.125,False,True,True,False
2,202606130417060004SMP,112.5,0.393882,112.5,0.358002,0.0,0.0,"112.5, 112.5, 112.5, 112.5",0.000,True,True,True,True
3,202606130420260005SMP,60.0,0.252463,60.0,0.246332,0.0,7.5,"67.5, 60.0, 60.0, 60.0",7.500,True,False,True,False
4,202606130422440006SMP,60.0,0.271998,60.0,0.258515,0.0,0.0,"60.0, 60.0, 60.0, 60.0",0.000,True,True,True,True
5,202606130426500007SMP,97.5,0.130005,97.5,0.140581,0.0,30.0,"97.5, 127.5, 97.5, 97.5",20.625,False,False,True,False
6,202606130430380008SMP,45.0,0.105586,45.0,0.135463,0.0,135.0,"60.0, 180.0, 45.0, 45.0",71.250,False,False,True,False
7,202606130433280009SMP,105.0,0.458371,105.0,0.445950,0.0,0.0,"105.0, 105.0, 105.0, 105.0",0.000,True,True,True,True
8,202606130434130010SMP,105.0,0.405844,105.0,0.418088,0.0,7.5,"112.5, 105.0, 105.0, 105.0",7.500,True,False,True,False
9,202606130437480012SMP,67.5,0.168268,67.5,0.126192,0.0,60.0,"67.5, 127.5, 67.5, 67.5",9.375,False,False,True,False


### Native HR robustness QC interpretation

Result classification:

This section supports native HR robustness metrics as useful QC features, but it does not support using the current native-only rule as the final NB12 classifier.

Evidence-supported observations:

- Three of ten recordings pass the combined window, subset, and filter robustness rule.
- `202606130417060004SMP` and `202606130422440006SMP` pass native robustness QC and are consistent with the expected native-specific group.
- `202606130433280009SMP` also passes native robustness QC, even though the working NB12 plan treats it as an audio-only candidate rather than a native-specific recording.
- `202606130426500007SMP` and `202606130430380008SMP` fail native robustness QC, consistent with the expected reject examples.
- Several expected native-specific recordings fail this simple native-only rule because of window instability, subset disagreement, or harmonic ambiguity.

What this supports:

- Window HR IQR, subset disagreement, and filter sensitivity are useful native QC descriptors.
- Native-only robustness can identify some stable examples and some reject examples.
- The final native class should not be assigned from this rule alone.

What this does not support:

- This result does not validate native beat timing.
- This result does not prove native-specific cardiac information for all expected native-specific recordings.
- This result does not justify promoting `hp_activity` to a standalone physiological signal.
- This result does not validate calibrated velocity, pressure, native Doppler indices, or clinical measurements.


## Existing audio/reference artifact discovery

### What this tests

This section tests which existing CSV artifacts contain current recording IDs and possible audio, HR, waveform, or QC columns.

### Why this matters

The native manifest did not find per-recording audio CSVs by direct filename search. This section checks whether useful reference information exists in batch-level summary tables or prior checkpoint outputs before NB12 attempts audio validation.

In [18]:
def list_candidate_csv_files(search_roots):
    """This function lists candidate CSV files in selected project output folders."""
    csv_files = []

    for root in search_roots:
        root = Path(root)
        if root.exists():
            csv_files.extend(sorted(root.rglob("*.csv")))

    return sorted(set(csv_files))


def safe_join_values(values, max_items=12):
    """This function joins values for compact display."""
    values = [str(value) for value in values if pd.notna(value)]
    values = sorted(set(values))

    if len(values) > max_items:
        shown = values[:max_items]
        return ", ".join(shown) + f", ... ({len(values)} total)"

    return ", ".join(values)


def summarize_csv_for_recording_matches(csv_path, recording_ids, max_rows_for_scan=50000):
    """This function summarizes whether a CSV contains recording IDs and possible reference columns."""
    csv_path = Path(csv_path)

    try:
        header = pd.read_csv(csv_path, nrows=0)
    except Exception as error:
        return {
            "csv_path": csv_path,
            "read_ok": False,
            "error": str(error),
            "n_rows_scanned": 0,
            "matched_recording_count": 0,
            "matched_recordings": "",
            "candidate_id_columns": "",
            "candidate_hr_columns": "",
            "candidate_audio_columns": "",
            "candidate_waveform_columns": "",
            "candidate_qc_columns": "",
            "all_columns": "",
        }

    columns = list(header.columns)
    lower_columns = {column: str(column).lower() for column in columns}

    candidate_id_columns = [
        column for column, lower in lower_columns.items()
        if any(token in lower for token in ["recording", "video", "file", "avi", "id"])
    ]
    candidate_hr_columns = [
        column for column, lower in lower_columns.items()
        if any(token in lower for token in ["hr", "bpm", "heart", "rate"])
    ]
    candidate_audio_columns = [
        column for column, lower in lower_columns.items()
        if any(token in lower for token in ["audio", "spectral", "wav", "sound"])
    ]
    candidate_waveform_columns = [
        column for column, lower in lower_columns.items()
        if any(token in lower for token in ["wave", "flow", "activity", "signal"])
    ]
    candidate_qc_columns = [
        column for column, lower in lower_columns.items()
        if any(token in lower for token in ["qc", "quality", "pass", "fail", "valid", "status"])
    ]

    matched_recordings = set()
    n_rows_scanned = 0

    try:
        chunk_iterator = pd.read_csv(
            csv_path,
            chunksize=5000,
            dtype=str,
            keep_default_na=False,
            encoding_errors="replace",
        )

        for chunk in chunk_iterator:
            n_rows_scanned += len(chunk)

            for column in chunk.columns:
                values = chunk[column].astype(str)
                for recording_id in recording_ids:
                    if values.str.contains(recording_id, regex=False).any():
                        matched_recordings.add(recording_id)

            if n_rows_scanned >= max_rows_for_scan:
                break

        read_ok = True
        error = ""
    except Exception as read_error:
        read_ok = False
        error = str(read_error)

    return {
        "csv_path": csv_path,
        "read_ok": read_ok,
        "error": error,
        "n_rows_scanned": n_rows_scanned,
        "matched_recording_count": len(matched_recordings),
        "matched_recordings": safe_join_values(matched_recordings),
        "candidate_id_columns": safe_join_values(candidate_id_columns),
        "candidate_hr_columns": safe_join_values(candidate_hr_columns),
        "candidate_audio_columns": safe_join_values(candidate_audio_columns),
        "candidate_waveform_columns": safe_join_values(candidate_waveform_columns),
        "candidate_qc_columns": safe_join_values(candidate_qc_columns),
        "all_columns": safe_join_values(columns, max_items=30),
    }

In [19]:
recording_ids = native_manifest["recording_id"].tolist()

candidate_csv_files = list_candidate_csv_files(
    [
        AUDIO_EXPORT_ROOT,
        SCREENING_EXPORT_ROOT,
        FEATURE_EXPORT_ROOT,
        PROJECT_ROOT / "scope_v2",
    ]
)

csv_discovery_rows = [
    summarize_csv_for_recording_matches(csv_path, recording_ids)
    for csv_path in candidate_csv_files
]

csv_discovery = pd.DataFrame(csv_discovery_rows)

matched_csv_discovery = csv_discovery.loc[
    csv_discovery["matched_recording_count"] > 0
].sort_values(["matched_recording_count", "csv_path"], ascending=[False, True])

reference_column_discovery = csv_discovery.loc[
    (csv_discovery["candidate_hr_columns"] != "")
    | (csv_discovery["candidate_audio_columns"] != "")
    | (csv_discovery["candidate_waveform_columns"] != "")
    | (csv_discovery["candidate_qc_columns"] != "")
].sort_values(["matched_recording_count", "csv_path"], ascending=[False, True])

print("Candidate CSV discovery summary")
print(f"CSV files checked: {len(csv_discovery)}")
print(
    "CSV files matching at least one recording ID: "
    f"{int((csv_discovery['matched_recording_count'] > 0).sum())} / {len(csv_discovery)}"
)
print(
    "CSV files with HR/audio/waveform/QC-like columns: "
    f"{len(reference_column_discovery)} / {len(csv_discovery)}"
)

display(
    matched_csv_discovery[
        [
            "csv_path",
            "n_rows_scanned",
            "matched_recording_count",
            "matched_recordings",
            "candidate_id_columns",
            "candidate_hr_columns",
            "candidate_audio_columns",
            "candidate_waveform_columns",
            "candidate_qc_columns",
        ]
    ].head(30)
)

display(
    reference_column_discovery[
        [
            "csv_path",
            "matched_recording_count",
            "candidate_id_columns",
            "candidate_hr_columns",
            "candidate_audio_columns",
            "candidate_waveform_columns",
            "candidate_qc_columns",
            "all_columns",
        ]
    ].head(40)
)

Candidate CSV discovery summary
CSV files checked: 135
CSV files matching at least one recording ID: 73 / 135
CSV files with HR/audio/waveform/QC-like columns: 107 / 135


,csv_path,n_rows_scanned,matched_recording_count,matched_recordings,candidate_id_columns,candidate_hr_columns,candidate_audio_columns,candidate_waveform_columns,candidate_qc_columns
16,D:\code\DopplerLab\scope_v2\reports\final_per_...,10,10,"202606130411060002SMP, 202606130413540003SMP, ...","gate6_audio_validation, recording_id",,"gate4_waveform_reproduced, gate6_audio_validation",gate4_waveform_reproduced,"gate6_audio_validation, main_failure_reason"
17,D:\code\DopplerLab\scope_v2\reports\gate0_inve...,10,10,"202606130411060002SMP, 202606130413540003SMP, ...","avi_duration_s, avi_path, recording_id",,,,inventory_status
18,D:\code\DopplerLab\scope_v2\reports\gate1_page...,10,10,"202606130411060002SMP, 202606130413540003SMP, ...","avi_duration_s, file_size_bytes, recording_id",,,,gate1_status
19,D:\code\DopplerLab\scope_v2\reports\gate2_reco...,160,10,"202606130411060002SMP, 202606130413540003SMP, ...","candidate_class, recording_id",,,,
20,D:\code\DopplerLab\scope_v2\reports\gate2_reco...,10,10,"202606130411060002SMP, 202606130413540003SMP, ...",recording_id,records_14_15_separated,,"n_signal_like_records, records_0_13_all_signal",gate2_status
21,D:\code\DopplerLab\scope_v2\reports\gate3_mapp...,10,10,"202606130411060002SMP, 202606130413540003SMP, ...",recording_id,sample_rate_hz,,,gate3_status
23,D:\code\DopplerLab\scope_v2\reports\gate4_wave...,20,10,"202606130411060002SMP, 202606130413540003SMP, ...",recording_id,"hr_iqr_bpm, median_hr_bpm, output_rate_hz",waveform_quality_score,waveform_quality_score,"gate4_status, waveform_quality_score"
24,D:\code\DopplerLab\scope_v2\reports\gate5_help...,10,10,"202606130411060002SMP, 202606130413540003SMP, ...",recording_id,"helper_hr_bpm, helper_implied_rate_hz, hr_abs_...",waveform_corr,waveform_corr,gate5_status
25,D:\code\DopplerLab\scope_v2\reports\gate6_inde...,10,10,"202606130411060002SMP, 202606130413540003SMP, ...",recording_id,"audio_hr_bpm, hr_abs_error_bpm, native_hr_bpm",audio_hr_bpm,,gate6_status
38,D:\code\DopplerLab\scope_v2\reverse\exports\pw...,10,10,"202606130411060002SMP, 202606130413540003SMP, ...",recording,HR_wave_bpm,HR_wave_bpm,HR_wave_bpm,


,csv_path,matched_recording_count,candidate_id_columns,candidate_hr_columns,candidate_audio_columns,candidate_waveform_columns,candidate_qc_columns,all_columns
16,D:\code\DopplerLab\scope_v2\reports\final_per_...,10,"gate6_audio_validation, recording_id",,"gate4_waveform_reproduced, gate6_audio_validation",gate4_waveform_reproduced,"gate6_audio_validation, main_failure_reason","allowed_interpretation, final_classification, ..."
17,D:\code\DopplerLab\scope_v2\reports\gate0_inve...,10,"avi_duration_s, avi_path, recording_id",,,,inventory_status,"avi_duration_s, avi_path, helper_csv_path, inv..."
18,D:\code\DopplerLab\scope_v2\reports\gate1_page...,10,"avi_duration_s, file_size_bytes, recording_id",,,,gate1_status,"avi_duration_s, divisible_by_1296, duration_ab..."
20,D:\code\DopplerLab\scope_v2\reports\gate2_reco...,10,recording_id,records_14_15_separated,,"n_signal_like_records, records_0_13_all_signal",gate2_status,"gate2_status, n_signal_like_records, recording..."
21,D:\code\DopplerLab\scope_v2\reports\gate3_mapp...,10,recording_id,sample_rate_hz,,,gate3_status,"boundary_over_within, cardiac_score, control_s..."
23,D:\code\DopplerLab\scope_v2\reports\gate4_wave...,10,recording_id,"hr_iqr_bpm, median_hr_bpm, output_rate_hz",waveform_quality_score,waveform_quality_score,"gate4_status, waveform_quality_score","control_periodicity_ac, filter_band_high_hz, f..."
24,D:\code\DopplerLab\scope_v2\reports\gate5_help...,10,recording_id,"helper_hr_bpm, helper_implied_rate_hz, hr_abs_...",waveform_corr,waveform_corr,gate5_status,"best_lag_s, gate5_status, helper_hr_bpm, helpe..."
25,D:\code\DopplerLab\scope_v2\reports\gate6_inde...,10,recording_id,"audio_hr_bpm, hr_abs_error_bpm, native_hr_bpm",audio_hr_bpm,,gate6_status,"agreement_class, audio_hr_bpm, gate6_status, h..."
38,D:\code\DopplerLab\scope_v2\reverse\exports\pw...,10,recording,HR_wave_bpm,HR_wave_bpm,HR_wave_bpm,,"HR_wave_bpm, dur_s, n_pages, periodicity_ac, r..."
39,D:\code\DopplerLab\scope_v2\reverse\exports\pw...,10,recording,hr_bpm,,"flow_intensity_mean, flow_intensity_std",,"duration_s, flow_intensity_mean, flow_intensit..."


### Existing audio/reference artifact discovery interpretation

Result classification:

This section identifies candidate reference tables for audio, HR, waveform, and QC comparisons.

Evidence-supported observations:

- The project contains `135` CSV files in the searched output folders.
- `73` CSV files contain at least one current native recording ID.
- `107` CSV files contain HR-, audio-, waveform-, or QC-like column names.
- Several `scope_v2` tables contain all ten recording IDs and columns such as `audio_hr_bpm`, `native_hr_bpm`, `hr_abs_error_bpm`, `gate6_status`, `signal_name`, and QC/status fields.
- Batch-level screening tables also contain audio/QC-like columns, even though direct recording-ID matching did not find the native IDs in those tables.

What this supports:

- NB12 can locate candidate audio/reference artifacts for later validation.
- The next step should inspect a small number of candidate tables before choosing an audio reference source.
- Prior helper and `scope_v2` outputs can guide experiment design.

What this does not support:

- This discovery step does not validate any helper-derived native conclusion.
- This discovery step does not prove that any table contains final ground truth.
- This discovery step does not validate native HR, beat timing, velocity, pressure, or Doppler indices.


## Candidate audio/reference table inspection

### What this tests

This section inspects a small set of candidate CSV tables that contain current recording IDs and audio-, HR-, waveform-, or QC-like columns.

### Why this matters

NB12 needs an audio-reference source before comparing native-derived HR with an external rhythm estimate. The previous discovery step found many possible tables, so this section narrows the candidates by checking table shape, column names, and preview rows.


In [20]:
def show_candidate_reference_tables(csv_discovery):
    """This function selects likely audio/reference tables for close inspection."""
    candidate_mask = (
        (csv_discovery["matched_recording_count"] == len(recording_ids))
        & (
            csv_discovery["candidate_audio_columns"].str.contains("audio", case=False, na=False)
            | csv_discovery["candidate_hr_columns"].str.contains("hr|bpm", case=False, regex=True, na=False)
        )
    )

    candidates = csv_discovery.loc[candidate_mask].copy()

    preferred_terms = [
        "gate6",
        "audio_validation",
        "final_native",
        "round5",
        "round6",
        "nb05_audio_qc_summary",
        "nb05_screening_report",
    ]

    candidates["priority_score"] = 0
    for index, term in enumerate(preferred_terms):
        candidates.loc[
            candidates["csv_path"].astype(str).str.contains(term, case=False, regex=False),
            "priority_score",
        ] += len(preferred_terms) - index

    candidates = candidates.sort_values(
        ["priority_score", "matched_recording_count", "csv_path"],
        ascending=[False, False, True],
    )

    return candidates


def inspect_csv_table(csv_path, max_rows=12):
    """This function loads a small preview of a candidate CSV table."""
    csv_path = Path(csv_path)
    table = pd.read_csv(csv_path)

    print("=" * 90)
    print(csv_path)
    print(f"shape: {table.shape}")
    print("columns:")
    print(list(table.columns))

    preview_columns = [
        column for column in table.columns
        if any(
            token in str(column).lower()
            for token in [
                "recording",
                "audio",
                "native",
                "hr",
                "bpm",
                "status",
                "class",
                "qc",
                "error",
                "signal",
            ]
        )
    ]

    if not preview_columns:
        preview_columns = list(table.columns[:12])

    display(table[preview_columns].head(max_rows))

    return table

In [21]:
candidate_reference_tables = show_candidate_reference_tables(csv_discovery)

print("Likely candidate reference tables")
display(
    candidate_reference_tables[
        [
            "csv_path",
            "matched_recording_count",
            "candidate_id_columns",
            "candidate_hr_columns",
            "candidate_audio_columns",
            "candidate_waveform_columns",
            "candidate_qc_columns",
        ]
    ].head(12)
)

tables_to_inspect = candidate_reference_tables["csv_path"].head(4).tolist()

inspected_reference_tables = {}
for csv_path in tables_to_inspect:
    inspected_reference_tables[str(csv_path)] = inspect_csv_table(csv_path)

Likely candidate reference tables


,csv_path,matched_recording_count,candidate_id_columns,candidate_hr_columns,candidate_audio_columns,candidate_waveform_columns,candidate_qc_columns
25,D:\code\DopplerLab\scope_v2\reports\gate6_inde...,10,recording_id,"audio_hr_bpm, hr_abs_error_bpm, native_hr_bpm",audio_hr_bpm,,gate6_status
71,D:\code\DopplerLab\scope_v2\round3_final_nativ...,10,recording_id,"audio_hr_bpm, hr_abs_error_bpm, inst_hr_iqr_bp...",audio_hr_bpm,signal_name,detector_status
72,D:\code\DopplerLab\scope_v2\round3_final_nativ...,10,"candidate_periodicity_score, recording_id","audio_hr_bpm, hr_abs_error_bpm, native_hr_bpm",audio_hr_bpm,"signal_name, signal_status","harmonic_status, signal_status"
73,D:\code\DopplerLab\scope_v2\round3_final_nativ...,10,"double_rate_candidate_bpm, half_rate_candidate...","audio_hr_bpm, double_rate_candidate_bpm, half_...","audio_hr_bpm, native_audio_abs_error_bpm",signal_name,harmonic_status
76,D:\code\DopplerLab\scope_v2\round3_final_nativ...,10,recording_id,"audio_hr_bpm, hr_abs_error_bpm, native_hr_bpm",audio_hr_bpm,best_native_signal,"beat_detector_status, final_native_status, har..."
78,D:\code\DopplerLab\scope_v2\round3_final_nativ...,10,"candidate_beats_control, candidate_periodicity...","audio_hr_bpm, hr_abs_error_bpm, native_hr_bpm",audio_hr_bpm,signal_name,control_status
93,D:\code\DopplerLab\scope_v2\round5_adversarial...,10,recording_id,"hr_abs_error_bpm, inst_hr_iqr_bpm, median_inst...",,,
95,D:\code\DopplerLab\scope_v2\round5_adversarial...,10,recording_id,"audio_hr_bpm, estimator_split_bpm, hr_autocorr...","audio_hr_bpm, audio_result",,harmonic_status
96,D:\code\DopplerLab\scope_v2\round5_adversarial...,10,recording_id,"audio_hr_bpm, filter_sensitivity_bpm, hr_abs_e...",audio_hr_bpm,,
97,D:\code\DopplerLab\scope_v2\round5_adversarial...,10,recording_id,subset_disagreement_bpm,,,


D:\code\DopplerLab\scope_v2\reports\gate6_independent_validation_summary.csv
shape: (10, 7)
columns:
['recording_id', 'native_hr_bpm', 'audio_hr_bpm', 'hr_abs_error_bpm', 'agreement_class', 'native_periodicity_ac', 'gate6_status']


,recording_id,native_hr_bpm,audio_hr_bpm,hr_abs_error_bpm,agreement_class,native_periodicity_ac,gate6_status
0,202606130411060002SMP,60.0,62.3,2.3,strong,0.234,pass
1,202606130413540003SMP,52.6,54.9,2.3,strong,0.126,pass
2,202606130417060004SMP,109.1,113.5,4.4,strong,0.398,pass
3,202606130420260005SMP,61.5,62.3,0.8,strong,0.317,pass
4,202606130422440006SMP,56.9,58.6,1.7,strong,0.220,pass
5,202606130426500007SMP,126.3,87.9,38.4,fail,0.122,review
6,202606130430380008SMP,114.3,109.9,4.4,strong,0.561,pass
7,202606130433280009SMP,105.3,109.7,4.4,strong,0.417,pass
8,202606130434130010SMP,105.3,106.2,0.9,strong,0.370,pass
9,202606130437480012SMP,150.0,65.9,84.1,fail,0.682,review


D:\code\DopplerLab\scope_v2\round3_final_native_audit\reports\round3_beat_detector_audit.csv
shape: (40, 14)
columns:
['recording_id', 'signal_name', 'detector_name', 'native_hr_prior_bpm', 'n_detected_beats', 'median_inst_hr_bpm', 'inst_hr_iqr_bpm', 'audio_hr_bpm', 'hr_abs_error_bpm', 'oversegmentation_flag', 'missed_cycle_flag', 'harmonic_flag', 'unstable_interval_flag', 'detector_status']


,recording_id,signal_name,native_hr_prior_bpm,median_inst_hr_bpm,inst_hr_iqr_bpm,audio_hr_bpm,hr_abs_error_bpm,detector_status
0,202606130411060002SMP,hp_activity,62.5,98.4,59.8,62.3,36.1,oversegments
1,202606130411060002SMP,hp_activity,62.5,63.8,10.3,62.3,1.5,good
2,202606130411060002SMP,hp_activity,62.5,61.9,8.0,62.3,0.4,good
3,202606130411060002SMP,hp_activity,62.5,61.9,4.3,62.3,0.4,good
4,202606130413540003SMP,hp_activity,54.1,100.0,56.1,54.9,45.1,oversegments
5,202606130413540003SMP,hp_activity,54.1,54.3,1.8,54.9,0.6,good
6,202606130413540003SMP,hp_activity,54.1,54.1,1.9,54.9,0.8,good
7,202606130413540003SMP,hp_activity,54.1,54.5,1.9,54.9,0.4,good
8,202606130417060004SMP,hp_activity,113.2,111.1,4.3,113.5,2.4,good
9,202606130417060004SMP,hp_activity,113.2,111.1,4.3,113.5,2.4,good


D:\code\DopplerLab\scope_v2\round3_final_native_audit\reports\round3_candidate_signal_rankings.csv
shape: (80, 13)
columns:
['recording_id', 'signal_name', 'native_rank', 'candidate_periodicity_score', 'best_control_p95', 'native_control_p_value', 'native_control_z_score', 'native_hr_bpm', 'audio_hr_bpm', 'hr_abs_error_bpm', 'agreement_class', 'harmonic_status', 'signal_status']


,recording_id,signal_name,native_rank,native_control_p_value,native_control_z_score,native_hr_bpm,audio_hr_bpm,hr_abs_error_bpm,agreement_class,harmonic_status,signal_status
0,202606130411060002SMP,hp_activity,8,0.0,4.40,62.5,62.3,0.2,strong,NaN,supported
1,202606130411060002SMP,mean_0_13,4,0.0,8.77,61.9,62.3,0.4,strong,NaN,supported
2,202606130411060002SMP,median_0_13,5,0.0,8.51,61.9,62.3,0.4,strong,NaN,supported
3,202606130411060002SMP,std_0_13,7,0.0,7.90,62.5,62.3,0.2,strong,NaN,supported
4,202606130411060002SMP,even_mean,1,0.0,14.24,61.9,62.3,0.4,strong,NaN,supported
5,202606130411060002SMP,odd_mean,6,0.0,7.96,63.2,62.3,0.9,strong,NaN,supported
6,202606130411060002SMP,pc1_0_13,2,0.0,9.91,61.9,62.3,0.4,strong,NaN,supported
7,202606130411060002SMP,records_14_15,3,0.0,9.15,61.9,62.3,0.4,strong,NaN,supported
8,202606130413540003SMP,hp_activity,7,0.0,2.87,54.1,54.9,0.8,strong,NaN,weak
9,202606130413540003SMP,mean_0_13,5,0.0,5.28,54.5,54.9,0.4,strong,NaN,supported


D:\code\DopplerLab\scope_v2\round3_final_native_audit\reports\round3_hr_harmonic_audit.csv
shape: (10, 14)
columns:
['recording_id', 'signal_name', 'hr_autocorr_bpm', 'hr_psd_bpm', 'hr_cepstrum_bpm', 'hr_peak_detector_bpm', 'hr_consensus_bpm', 'half_rate_candidate_bpm', 'double_rate_candidate_bpm', 'audio_hr_bpm', 'native_audio_abs_error_bpm', 'native_estimator_agreement', 'harmonic_status', 'harmonic_evidence']


,recording_id,signal_name,hr_autocorr_bpm,hr_psd_bpm,hr_cepstrum_bpm,hr_peak_detector_bpm,hr_consensus_bpm,half_rate_candidate_bpm,double_rate_candidate_bpm,audio_hr_bpm,native_audio_abs_error_bpm,native_estimator_agreement,harmonic_status
0,202606130411060002SMP,hp_activity,62.5,60.0,61.9,130.4,61.2,31.2,125.0,62.3,1.0,2.5,clean
1,202606130413540003SMP,hp_activity,54.1,52.5,150.0,113.2,53.3,27.0,108.1,54.9,1.6,1.6,clean
2,202606130417060004SMP,hp_activity,113.2,112.5,113.2,111.1,112.9,56.6,226.4,113.5,0.6,0.7,clean
3,202606130420260005SMP,hp_activity,64.5,63.8,61.2,133.3,64.1,32.3,129.0,62.3,1.8,0.8,clean
4,202606130422440006SMP,hp_activity,58.3,60.0,57.1,141.2,59.1,29.1,116.5,58.6,0.5,1.7,clean
5,202606130426500007SMP,hp_activity,64.5,146.2,62.5,125.0,105.4,32.3,129.0,87.9,17.5,81.7,harmonic_ambiguous
6,202606130430380008SMP,hp_activity,200.0,183.8,125.0,125.0,191.9,100.0,400.0,109.9,82.0,16.2,clean
7,202606130433280009SMP,hp_activity,109.1,108.8,109.1,109.1,108.9,54.5,218.2,109.7,0.8,0.3,clean
8,202606130434130010SMP,hp_activity,107.1,108.8,54.1,110.1,107.9,53.6,214.3,106.2,1.7,1.6,clean
9,202606130437480012SMP,hp_activity,65.2,67.5,66.7,122.4,66.4,32.6,130.4,65.9,0.5,2.3,clean


### Candidate audio/reference table inspection interpretation

Result classification:

This section identifies candidate audio-reference and helper-audit tables, but it does not promote any helper-derived classification to final NB12 evidence.

Evidence-supported observations:

- Several candidate tables contain all ten current recording IDs.
- `gate6_independent_validation_summary.csv` contains `native_hr_bpm`, `audio_hr_bpm`, `hr_abs_error_bpm`, `agreement_class`, `native_periodicity_ac`, and `gate6_status`.
- `round3_beat_detector_audit.csv` contains beat-detector outcomes for multiple detector variants and shows that some detector choices oversegment while others produce lower HR error.
- `round3_candidate_signal_rankings.csv` contains multiple candidate native signals and control-style metrics, but it is a helper-derived ranking table.
- `round3_hr_harmonic_audit.csv` contains multiple native HR estimators, `audio_hr_bpm`, native-audio error, estimator agreement, and harmonic status.
- Candidate helper outputs do not fully agree with the current NB12 working classification. For example, some tables mark `202606130430380008SMP` as audio-agreeing or pass-like, even though the NB12 plan expects it to remain a reject example.
- `202606130437480012SMP` also shows table-dependent behavior, which is consistent with its borderline status.

What this supports:

- Existing tables can provide candidate audio HR references for comparison.
- `audio_hr_bpm` columns are useful as external reference candidates, especially when they appear in compact all-recording tables.
- Helper classifications and helper native-status labels require caution and should not be copied directly into NB12 conclusions.

What this does not support:

- This section does not validate helper-derived native classifications.
- This section does not prove that `hp_activity` is native-specific in every audio-agreeing recording.
- This section does not validate native beat timing, calibrated velocity, pressure, native Doppler indices, or clinical interpretation.
- This section does not resolve final native class labels.


## Candidate audio-reference comparison

### What this tests

This section tests whether the NB12-reproduced `hp_activity` HR estimates agree with a candidate audio HR reference table.

### Why this matters

Native-only QC was informative but not decisive. Comparing NB12-reproduced native HR against candidate audio HR can show whether the native timing/QC signal is broadly consistent with an independent AVI-audio-derived rhythm estimate.


In [22]:
def find_csv_by_name(project_root, file_name):
    """This function finds a CSV file by exact filename under the project root."""
    matches = sorted(Path(project_root).rglob(file_name))

    if len(matches) == 0:
        raise FileNotFoundError(f"Could not find {file_name} under {project_root}")

    if len(matches) > 1:
        print(f"Multiple matches found for {file_name}. Using the first match:")
        for match in matches:
            print(match)

    return matches[0]


def load_candidate_audio_reference(project_root):
    """This function loads a compact candidate audio-reference table for NB12 comparison."""
    reference_path = find_csv_by_name(
        project_root,
        "round3_hr_harmonic_audit.csv",
    )

    reference = pd.read_csv(reference_path)

    required_columns = ["recording_id", "audio_hr_bpm"]
    missing_columns = [
        column for column in required_columns
        if column not in reference.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Candidate audio reference is missing required columns: {missing_columns}"
        )

    keep_columns = [
        column for column in [
            "recording_id",
            "signal_name",
            "audio_hr_bpm",
            "hr_consensus_bpm",
            "native_audio_abs_error_bpm",
            "native_estimator_agreement",
            "harmonic_status",
        ]
        if column in reference.columns
    ]

    reference = reference[keep_columns].copy()
    reference = reference.drop_duplicates(subset=["recording_id"]).reset_index(drop=True)

    return reference_path, reference


def build_nb12_audio_comparison(native_hr_summary, native_hr_robustness, audio_reference):
    """This function compares NB12-reproduced native HR estimates with candidate audio HR."""
    comparison = native_hr_summary.merge(
        native_hr_robustness[
            [
                "recording_id",
                "subset_disagreement_bpm",
                "filter_sensitivity_bpm",
                "passes_native_robustness_qc",
            ]
        ],
        on="recording_id",
        how="left",
    )

    comparison = comparison.merge(
        audio_reference,
        on="recording_id",
        how="left",
    )

    comparison["global_native_audio_abs_error_bpm"] = (
        comparison["global_native_hr_bpm"] - comparison["audio_hr_bpm"]
    ).abs()

    comparison["window_median_audio_abs_error_bpm"] = (
        comparison["window_hr_median_bpm"] - comparison["audio_hr_bpm"]
    ).abs()

    comparison["global_audio_agreement_candidate"] = np.select(
        [
            comparison["global_native_audio_abs_error_bpm"] <= 5,
            comparison["global_native_audio_abs_error_bpm"] <= 10,
        ],
        [
            "within_5_bpm",
            "within_10_bpm",
        ],
        default="outside_10_bpm",
    )

    expected_working_class = {
        "202606130411060002SMP": "native_specific",
        "202606130413540003SMP": "native_specific",
        "202606130417060004SMP": "native_specific",
        "202606130420260005SMP": "native_specific",
        "202606130422440006SMP": "native_specific",
        "202606130437480012SMP": "native_specific_borderline",
        "202606130433280009SMP": "audio_only_candidate",
        "202606130434130010SMP": "audio_only_candidate",
        "202606130426500007SMP": "reject",
        "202606130430380008SMP": "reject",
    }

    comparison["nb12_working_class_from_plan"] = comparison["recording_id"].map(expected_working_class)

    display_columns = [
        "recording_id",
        "nb12_working_class_from_plan",
        "global_native_hr_bpm",
        "window_hr_median_bpm",
        "audio_hr_bpm",
        "global_native_audio_abs_error_bpm",
        "window_median_audio_abs_error_bpm",
        "global_audio_agreement_candidate",
        "window_hr_iqr_bpm",
        "subset_disagreement_bpm",
        "filter_sensitivity_bpm",
        "passes_native_robustness_qc",
        "harmonic_status",
    ]

    display_columns = [
        column for column in display_columns
        if column in comparison.columns
    ]

    return comparison[display_columns].sort_values("recording_id").reset_index(drop=True)

In [23]:
candidate_audio_reference_path, candidate_audio_reference = load_candidate_audio_reference(PROJECT_ROOT)

nb12_audio_comparison = build_nb12_audio_comparison(
    native_hr_summary,
    native_hr_robustness,
    candidate_audio_reference,
)

print("Candidate audio-reference comparison")
print(f"Candidate audio reference path: {candidate_audio_reference_path}")
print(f"Recordings with candidate audio HR: {int(nb12_audio_comparison['audio_hr_bpm'].notna().sum())} / {len(nb12_audio_comparison)}")
print(
    "Recordings with NB12 global native HR within 5 bpm of candidate audio HR: "
    f"{int((nb12_audio_comparison['global_native_audio_abs_error_bpm'] <= 5).sum())} / {len(nb12_audio_comparison)}"
)
print(
    "Recordings with NB12 global native HR outside 10 bpm of candidate audio HR: "
    f"{int((nb12_audio_comparison['global_native_audio_abs_error_bpm'] > 10).sum())} / {len(nb12_audio_comparison)}"
)

display(nb12_audio_comparison)

Candidate audio-reference comparison
Candidate audio reference path: D:\code\DopplerLab\scope_v2\round3_final_native_audit\reports\round3_hr_harmonic_audit.csv
Recordings with candidate audio HR: 10 / 10
Recordings with NB12 global native HR within 5 bpm of candidate audio HR: 8 / 10
Recordings with NB12 global native HR outside 10 bpm of candidate audio HR: 1 / 10


,recording_id,nb12_working_class_from_plan,global_native_hr_bpm,window_hr_median_bpm,audio_hr_bpm,global_native_audio_abs_error_bpm,window_median_audio_abs_error_bpm,global_audio_agreement_candidate,window_hr_iqr_bpm,subset_disagreement_bpm,filter_sensitivity_bpm,passes_native_robustness_qc,harmonic_status
0,202606130411060002SMP,native_specific,60.0,63.75,62.3,2.3,1.45,within_5_bpm,61.875,67.5,0.0,False,clean
1,202606130413540003SMP,native_specific,52.5,52.50,54.9,2.4,2.40,within_5_bpm,13.125,0.0,0.0,False,clean
2,202606130417060004SMP,native_specific,112.5,112.50,113.5,1.0,1.00,within_5_bpm,0.000,0.0,0.0,True,clean
3,202606130420260005SMP,native_specific,60.0,67.50,62.3,2.3,5.20,within_5_bpm,7.500,7.5,0.0,False,clean
4,202606130422440006SMP,native_specific,60.0,60.00,58.6,1.4,1.40,within_5_bpm,0.000,0.0,0.0,True,clean
5,202606130426500007SMP,reject,97.5,112.50,87.9,9.6,24.60,within_10_bpm,20.625,30.0,0.0,False,harmonic_ambiguous
6,202606130430380008SMP,reject,45.0,105.00,109.9,64.9,4.90,outside_10_bpm,71.250,135.0,0.0,False,clean
7,202606130433280009SMP,audio_only_candidate,105.0,105.00,109.7,4.7,4.70,within_5_bpm,0.000,0.0,0.0,True,clean
8,202606130434130010SMP,audio_only_candidate,105.0,105.00,106.2,1.2,1.20,within_5_bpm,7.500,7.5,0.0,False,clean
9,202606130437480012SMP,native_specific_borderline,67.5,63.75,65.9,1.6,2.15,within_5_bpm,9.375,60.0,0.0,False,clean


### Candidate audio-reference comparison interpretation

Result classification:

This section supports broad agreement between the NB12-reproduced `hp_activity` global HR estimate and a candidate audio HR reference, but it does not support final native class assignment by audio agreement alone.

Evidence-supported observations:

- Candidate audio HR was available for all ten recordings.
- Eight of ten recordings had NB12 global native HR within `5 bpm` of candidate audio HR.
- One recording, `202606130430380008SMP`, had NB12 global native HR outside `10 bpm` of candidate audio HR.
- `202606130426500007SMP` had global native HR within `10 bpm`, but had weaker evidence because the window-median error was high and the harmonic status was `harmonic_ambiguous`.
- Most planned native-specific recordings showed good global native-audio agreement, even when native-only robustness QC did not pass.
- `202606130433280009SMP` showed good native-audio agreement and passed native robustness QC, but the NB12 working plan still treats it as an audio-only candidate, so audio agreement alone is not enough for native-specific adoption.
- `202606130437480012SMP` showed good global native-audio agreement but failed native robustness QC, consistent with a borderline interpretation.

What this supports:

- `hp_activity` can produce candidate HR estimates that often agree with candidate audio HR.
- Candidate audio comparison is useful for NB12 timing/QC interpretation.
- Native-audio agreement should be included as one evidence feature in the final native signal classification table.

What this does not support:

- This result does not validate native beat timing.
- This result does not prove that audio-agreeing native HR is native-specific.
- This result does not justify adopting `hp_activity` as a standalone physiological waveform.
- This result does not validate calibrated velocity, pressure, native Doppler indices, or clinical interpretation.
- The candidate audio table remains a reference artifact, not ground truth.


## Native signal classification synthesis

### What this tests

This section combines NB12-reproduced evidence into a cautious native signal classification table.

### Why this matters

The previous sections show that no single feature is decisive. Fixed-page structure, counter integrity, `hp_activity` reproducibility, native robustness, and candidate audio agreement need to be summarized together before assigning final native-file utility classes.



In [24]:
def classify_native_signal_evidence(row):
    """This function assigns a cautious NB12 evidence classification from reproduced QC features."""
    plan_class = row["nb12_working_class_from_plan"]

    global_error = row["global_native_audio_abs_error_bpm"]
    window_error = row["window_median_audio_abs_error_bpm"]
    window_iqr = row["window_hr_iqr_bpm"]
    subset_disagreement = row["subset_disagreement_bpm"]
    robustness_pass = bool(row["passes_native_robustness_qc"])
    harmonic_status = str(row.get("harmonic_status", "")).lower()

    global_audio_agrees = np.isfinite(global_error) and global_error <= 5
    window_audio_agrees = np.isfinite(window_error) and window_error <= 5
    any_audio_agrees = global_audio_agrees or window_audio_agrees

    strong_instability = (
        (np.isfinite(window_iqr) and window_iqr >= 20)
        or (np.isfinite(subset_disagreement) and subset_disagreement >= 30)
    )
    harmonic_ambiguous = "ambiguous" in harmonic_status

    if plan_class == "native_specific":
        if global_audio_agrees and robustness_pass:
            return "native_specific_supported"
        if global_audio_agrees and not robustness_pass:
            return "native_specific_with_qc_caveat"
        if any_audio_agrees:
            return "native_specific_candidate_with_method_caveat"
        return "native_specific_not_supported_by_current_summary"

    if plan_class == "native_specific_borderline":
        if any_audio_agrees and not robustness_pass:
            return "borderline_native_specific_with_qc_caveat"
        if any_audio_agrees and robustness_pass:
            return "borderline_native_specific_supported"
        return "borderline_not_supported_by_current_summary"

    if plan_class == "audio_only_candidate":
        if any_audio_agrees and robustness_pass:
            return "audio_only_candidate_with_native_agreement_conflict"
        if any_audio_agrees:
            return "audio_only_candidate_supported"
        return "audio_only_candidate_not_supported_by_current_summary"

    if plan_class == "reject":
        if strong_instability or harmonic_ambiguous or not any_audio_agrees:
            return "reject_supported"
        return "reject_with_audio_agreement_caveat"

    return "unclassified"


def build_native_signal_classification(audio_comparison):
    """This function builds the NB12 native signal classification synthesis table."""
    classification = audio_comparison.copy()

    classification["global_audio_agrees_within_5_bpm"] = (
        classification["global_native_audio_abs_error_bpm"] <= 5
    )
    classification["window_audio_agrees_within_5_bpm"] = (
        classification["window_median_audio_abs_error_bpm"] <= 5
    )
    classification["strong_instability_flag"] = (
        (classification["window_hr_iqr_bpm"] >= 20)
        | (classification["subset_disagreement_bpm"] >= 30)
    )
    classification["harmonic_ambiguous_flag"] = (
        classification["harmonic_status"].astype(str).str.contains("ambiguous", case=False, na=False)
    )

    classification["nb12_evidence_class"] = classification.apply(
        classify_native_signal_evidence,
        axis=1,
    )

    classification["allowed_claim"] = "experimental native timing/QC sidecar"
    classification["forbidden_claim"] = (
        "calibrated velocity, pressure, native Doppler indices, clinical interpretation"
    )

    display_columns = [
        "recording_id",
        "nb12_working_class_from_plan",
        "nb12_evidence_class",
        "global_native_hr_bpm",
        "audio_hr_bpm",
        "global_native_audio_abs_error_bpm",
        "window_median_audio_abs_error_bpm",
        "passes_native_robustness_qc",
        "strong_instability_flag",
        "harmonic_ambiguous_flag",
        "harmonic_status",
        "allowed_claim",
        "forbidden_claim",
    ]

    return classification[display_columns].sort_values("recording_id").reset_index(drop=True)


In [25]:
nb12_native_signal_classification = build_native_signal_classification(nb12_audio_comparison)

print("NB12 native signal classification synthesis")
print(f"Recordings classified: {len(nb12_native_signal_classification)}")
print("Evidence class counts:")
display(nb12_native_signal_classification["nb12_evidence_class"].value_counts().rename_axis("nb12_evidence_class").reset_index(name="n"))

display(nb12_native_signal_classification)

NB12 native signal classification synthesis
Recordings classified: 10
Evidence class counts:


,nb12_evidence_class,n
0,native_specific_with_qc_caveat,3
1,native_specific_supported,2
2,reject_supported,2
3,audio_only_candidate_with_native_agreement_con...,1
4,audio_only_candidate_supported,1
5,borderline_native_specific_with_qc_caveat,1


,recording_id,nb12_working_class_from_plan,nb12_evidence_class,global_native_hr_bpm,audio_hr_bpm,global_native_audio_abs_error_bpm,window_median_audio_abs_error_bpm,passes_native_robustness_qc,strong_instability_flag,harmonic_ambiguous_flag,harmonic_status,allowed_claim,forbidden_claim
0,202606130411060002SMP,native_specific,native_specific_with_qc_caveat,60.0,62.3,2.3,1.45,False,True,False,clean,experimental native timing/QC sidecar,"calibrated velocity, pressure, native Doppler ..."
1,202606130413540003SMP,native_specific,native_specific_with_qc_caveat,52.5,54.9,2.4,2.40,False,False,False,clean,experimental native timing/QC sidecar,"calibrated velocity, pressure, native Doppler ..."
2,202606130417060004SMP,native_specific,native_specific_supported,112.5,113.5,1.0,1.00,True,False,False,clean,experimental native timing/QC sidecar,"calibrated velocity, pressure, native Doppler ..."
3,202606130420260005SMP,native_specific,native_specific_with_qc_caveat,60.0,62.3,2.3,5.20,False,False,False,clean,experimental native timing/QC sidecar,"calibrated velocity, pressure, native Doppler ..."
4,202606130422440006SMP,native_specific,native_specific_supported,60.0,58.6,1.4,1.40,True,False,False,clean,experimental native timing/QC sidecar,"calibrated velocity, pressure, native Doppler ..."
5,202606130426500007SMP,reject,reject_supported,97.5,87.9,9.6,24.60,False,True,True,harmonic_ambiguous,experimental native timing/QC sidecar,"calibrated velocity, pressure, native Doppler ..."
6,202606130430380008SMP,reject,reject_supported,45.0,109.9,64.9,4.90,False,True,False,clean,experimental native timing/QC sidecar,"calibrated velocity, pressure, native Doppler ..."
7,202606130433280009SMP,audio_only_candidate,audio_only_candidate_with_native_agreement_con...,105.0,109.7,4.7,4.70,True,False,False,clean,experimental native timing/QC sidecar,"calibrated velocity, pressure, native Doppler ..."
8,202606130434130010SMP,audio_only_candidate,audio_only_candidate_supported,105.0,106.2,1.2,1.20,False,False,False,clean,experimental native timing/QC sidecar,"calibrated velocity, pressure, native Doppler ..."
9,202606130437480012SMP,native_specific_borderline,borderline_native_specific_with_qc_caveat,67.5,65.9,1.6,2.15,False,True,False,clean,experimental native timing/QC sidecar,"calibrated velocity, pressure, native Doppler ..."


### Native signal classification synthesis interpretation

Result classification:

This section supports a cautious NB12 signal classification, but it also shows why the surviving native signal should remain a timing/QC sidecar rather than a standalone physiological waveform.

Evidence-supported observations:

- Ten recordings were classified using NB12-reproduced native evidence and candidate audio-reference comparison.
- Two recordings, `202606130417060004SMP` and `202606130422440006SMP`, were classified as `native_specific_supported`.
- Three planned native-specific recordings were classified as `native_specific_with_qc_caveat`, meaning candidate audio agreement was good but native robustness QC was not fully stable.
- `202606130437480012SMP` was classified as `borderline_native_specific_with_qc_caveat`, consistent with the planned borderline status.
- `202606130426500007SMP` and `202606130430380008SMP` were classified as `reject_supported`.
- `202606130433280009SMP` was classified as `audio_only_candidate_with_native_agreement_conflict`, because it showed both audio agreement and native robustness despite the planned audio-only classification.
- `202606130434130010SMP` was classified as `audio_only_candidate_supported`.

What this supports:

- `hp_activity` has useful experimental timing/QC value in selected recordings.
- Final interpretation needs multiple evidence features, not a single HR match.
- Native-specific use should be limited to timing/QC support, with caveats documented per recording.
- Reject examples remain useful because they show instability, harmonic ambiguity, or native-audio disagreement.

What this does not support:

- This result does not validate native beat timing.
- This result does not validate a native velocity envelope.
- This result does not support native PSV, EDV, RI, PI, VTI, pressure, or clinical interpretation.
- This result does not prove that every audio-agreeing native signal is physiologically specific.


## Closure test - high-rate Doppler interpretation

### What this tests

This section tests whether records `0..13` support a useful high-rate `3500 Hz` Doppler-like complex stream when interpreted as seven `(I, Q)` pairs per page.

### Why this matters

Earlier helper work explored a `7 * 500 Hz = 3500 Hz` interpretation. NB12 needs to decide whether that interpretation survives as native Doppler evidence or should be closed for the current data.


In [26]:
def extract_candidate_iq_pairs_from_live_values(live_values):
    """This function interprets 14 live float values as seven candidate complex I/Q pairs."""
    live_values = np.asarray(live_values, dtype=float)

    if live_values.ndim != 2 or live_values.shape[1] < 14:
        raise ValueError("Expected live_values with shape (n_pages, at least 14).")

    i_values = live_values[:, 0:14:2]
    q_values = live_values[:, 1:14:2]
    complex_pairs = i_values + 1j * q_values

    return complex_pairs


def summarize_candidate_high_rate_phase(complex_pairs):
    """This function summarizes phase behavior inside candidate seven-pair pages."""
    complex_pairs = np.asarray(complex_pairs)

    magnitude = np.abs(complex_pairs)
    valid_pair = np.isfinite(complex_pairs.real) & np.isfinite(complex_pairs.imag) & (magnitude > 0)

    step_valid = valid_pair[:, 1:] & valid_pair[:, :-1]
    step_phase = np.angle(complex_pairs[:, 1:] * np.conj(complex_pairs[:, :-1]))

    valid_steps = step_phase[step_valid]

    net_valid = valid_pair[:, -1] & valid_pair[:, 0]
    net_phase = np.angle(complex_pairs[:, -1] * np.conj(complex_pairs[:, 0]))
    valid_net_phase = net_phase[net_valid]

    if len(valid_steps) == 0:
        return {
            "valid_step_count": 0,
            "mean_step_phase_rad": np.nan,
            "median_step_phase_rad": np.nan,
            "mean_abs_step_phase_rad": np.nan,
            "forward_step_fraction": np.nan,
            "valid_net_page_count": 0,
            "mean_net_phase_rad": np.nan,
            "median_net_phase_rad": np.nan,
            "mean_abs_net_phase_rad": np.nan,
        }

    return {
        "valid_step_count": int(len(valid_steps)),
        "mean_step_phase_rad": float(np.mean(valid_steps)),
        "median_step_phase_rad": float(np.median(valid_steps)),
        "mean_abs_step_phase_rad": float(np.mean(np.abs(valid_steps))),
        "forward_step_fraction": float(np.mean(valid_steps > 0)),
        "valid_net_page_count": int(len(valid_net_phase)),
        "mean_net_phase_rad": float(np.mean(valid_net_phase)) if len(valid_net_phase) else np.nan,
        "median_net_phase_rad": float(np.median(valid_net_phase)) if len(valid_net_phase) else np.nan,
        "mean_abs_net_phase_rad": float(np.mean(np.abs(valid_net_phase))) if len(valid_net_phase) else np.nan,
    }


def estimate_high_rate_magnitude_hr(complex_pairs):
    """This function estimates HR from the candidate high-rate magnitude stream."""
    magnitude_stream = np.abs(complex_pairs).reshape(-1)

    if len(magnitude_stream) == 0:
        return np.nan, np.nan

    magnitude_stream = np.nan_to_num(magnitude_stream, nan=np.nanmedian(magnitude_stream))
    high_rate_hz = PAGE_RATE_HZ * 7.0

    return estimate_native_hr_for_activity(
        magnitude_stream,
        source_rate_hz=high_rate_hz,
        env_rate_hz=ENV_RATE_HZ,
    )


def summarize_high_rate_doppler_closure(signals, audio_comparison):
    """This function summarizes high-rate Doppler closure evidence for each recording."""
    audio_lookup = audio_comparison.set_index("recording_id")

    rows = []
    for recording_id, signal_dict in signals.items():
        live_values = signal_dict["live_values"]
        complex_pairs = extract_candidate_iq_pairs_from_live_values(live_values)

        phase_summary = summarize_candidate_high_rate_phase(complex_pairs)
        high_rate_hr_bpm, high_rate_peak_fraction = estimate_high_rate_magnitude_hr(complex_pairs)

        native_500_hr_bpm = float(audio_lookup.loc[recording_id, "global_native_hr_bpm"])
        audio_hr_bpm = float(audio_lookup.loc[recording_id, "audio_hr_bpm"])

        rows.append(
            {
                "recording_id": recording_id,
                "candidate_high_rate_hz": PAGE_RATE_HZ * 7.0,
                "high_rate_magnitude_hr_bpm": high_rate_hr_bpm,
                "high_rate_peak_fraction": high_rate_peak_fraction,
                "base_500hz_hp_activity_hr_bpm": native_500_hr_bpm,
                "candidate_audio_hr_bpm": audio_hr_bpm,
                "high_rate_audio_abs_error_bpm": abs(high_rate_hr_bpm - audio_hr_bpm)
                if np.isfinite(high_rate_hr_bpm) and np.isfinite(audio_hr_bpm)
                else np.nan,
                "base_500hz_audio_abs_error_bpm": abs(native_500_hr_bpm - audio_hr_bpm)
                if np.isfinite(native_500_hr_bpm) and np.isfinite(audio_hr_bpm)
                else np.nan,
                **phase_summary,
            }
        )

    summary = pd.DataFrame(rows).sort_values("recording_id").reset_index(drop=True)

    summary["forward_fraction_near_half"] = (
        (summary["forward_step_fraction"] >= 0.45)
        & (summary["forward_step_fraction"] <= 0.55)
    )
    summary["net_phase_near_zero"] = summary["mean_net_phase_rad"].abs() < 0.10
    summary["high_rate_improves_audio_error"] = (
        summary["high_rate_audio_abs_error_bpm"]
        < summary["base_500hz_audio_abs_error_bpm"]
    )

    return summary

In [27]:
high_rate_closure_summary = summarize_high_rate_doppler_closure(
    hp_activity_signals,
    nb12_audio_comparison,
)

print("High-rate Doppler closure summary")
print(f"Recordings checked: {len(high_rate_closure_summary)}")
print(
    "Recordings with forward phase-step fraction near 0.5: "
    f"{int(high_rate_closure_summary['forward_fraction_near_half'].sum())} / {len(high_rate_closure_summary)}"
)
print(
    "Recordings with mean net page phase near zero: "
    f"{int(high_rate_closure_summary['net_phase_near_zero'].sum())} / {len(high_rate_closure_summary)}"
)
print(
    "Recordings where high-rate magnitude HR improves over base 500 Hz HR: "
    f"{int(high_rate_closure_summary['high_rate_improves_audio_error'].sum())} / {len(high_rate_closure_summary)}"
)

display(
    high_rate_closure_summary[
        [
            "recording_id",
            "candidate_high_rate_hz",
            "forward_step_fraction",
            "mean_step_phase_rad",
            "mean_abs_step_phase_rad",
            "mean_net_phase_rad",
            "mean_abs_net_phase_rad",
            "forward_fraction_near_half",
            "net_phase_near_zero",
            "high_rate_magnitude_hr_bpm",
            "base_500hz_hp_activity_hr_bpm",
            "candidate_audio_hr_bpm",
            "high_rate_audio_abs_error_bpm",
            "base_500hz_audio_abs_error_bpm",
            "high_rate_improves_audio_error",
        ]
    ]
)

High-rate Doppler closure summary
Recordings checked: 10
Recordings with forward phase-step fraction near 0.5: 3 / 10
Recordings with mean net page phase near zero: 10 / 10
Recordings where high-rate magnitude HR improves over base 500 Hz HR: 1 / 10


,recording_id,candidate_high_rate_hz,forward_step_fraction,mean_step_phase_rad,mean_abs_step_phase_rad,mean_net_phase_rad,mean_abs_net_phase_rad,forward_fraction_near_half,net_phase_near_zero,high_rate_magnitude_hr_bpm,base_500hz_hp_activity_hr_bpm,candidate_audio_hr_bpm,high_rate_audio_abs_error_bpm,base_500hz_audio_abs_error_bpm,high_rate_improves_audio_error
0,202606130411060002SMP,3500.0,0.473302,-0.001462,0.013881,-0.001556,0.044767,True,True,60.0,60.0,62.3,2.3,2.3,False
1,202606130413540003SMP,3500.0,0.465476,0.000662,0.014440,0.001830,0.060782,True,True,52.5,52.5,54.9,2.4,2.4,False
2,202606130417060004SMP,3500.0,0.414225,-0.000344,0.010636,-0.000585,0.046641,False,True,112.5,112.5,113.5,1.0,1.0,False
3,202606130420260005SMP,3500.0,0.444913,-0.000158,0.016922,0.009541,0.089232,False,True,60.0,60.0,62.3,2.3,2.3,False
4,202606130422440006SMP,3500.0,0.387308,0.000805,0.018495,0.002624,0.092262,False,True,60.0,60.0,58.6,1.4,1.4,False
5,202606130426500007SMP,3500.0,0.564555,-0.000040,0.025200,0.010034,0.126609,False,True,90.0,97.5,87.9,2.1,9.6,True
6,202606130430380008SMP,3500.0,0.517421,-0.000871,0.135391,-0.006936,0.175615,True,True,45.0,45.0,109.9,64.9,64.9,False
7,202606130433280009SMP,3500.0,0.334752,-0.000016,0.009383,-0.000093,0.053565,False,True,45.0,105.0,109.7,64.7,4.7,False
8,202606130434130010SMP,3500.0,0.354851,-0.000194,0.008931,-0.001161,0.049332,False,True,45.0,105.0,106.2,61.2,1.2,False
9,202606130437480012SMP,3500.0,0.410517,-0.004095,0.179385,-0.000958,0.200991,False,True,45.0,67.5,65.9,20.9,1.6,False


### High-rate Doppler closure interpretation

Result classification:

This section supports closing the `3500 Hz` high-rate native Doppler interpretation for the current data.

Evidence-supported observations:

- All ten recordings had mean net page phase near zero.
- Only three of ten recordings had forward phase-step fraction near `0.5`; values near `0.5` are more consistent with no stable directional phase preference than with a directional Doppler-like phase rotation.
- The mean step phase was close to zero in all recordings.
- The high-rate magnitude HR matched the base `500 Hz` `hp_activity` HR in several recordings, suggesting it often collapses back to the page-rate activity signal rather than adding independent high-rate Doppler information.
- The high-rate magnitude HR improved audio-HR error in only one recording, `202606130426500007SMP`, which is also a reject example with harmonic ambiguity.
- In `202606130433280009SMP`, `202606130434130010SMP`, and `202606130437480012SMP`, the high-rate magnitude HR degraded badly relative to the base `500 Hz` signal.

What this supports:

- The seven-pair `3500 Hz` interpretation does not provide consistent directional phase evidence.
- The high-rate magnitude variant does not improve the native timing/QC signal in a reliable way.
- High-rate native Doppler velocity extraction should be closed for the current data.

What this does not support:

- This result does not prove that no hidden native Doppler representation exists anywhere in the export.
- This result does not validate calibrated native velocity, native Doppler indices, pressure, native spectrogram recovery, or clinical interpretation.
- This result does not invalidate the lower-rate `hp_activity` timing/QC sidecar.


## Native file utility matrix

### What this tests

This section creates a final utility matrix for native-file outputs and candidate extraction paths.

### Why this matters

NB12 is a closure notebook. It needs a compact evidence ledger that states which native-derived outputs should be adopted, kept as future candidates, rejected, or closed.


In [28]:
def build_native_file_utility_matrix():
    """This function builds the NB12 native-file utility and claim-boundary matrix."""
    rows = [
        {
            "output_name": "PW page timebase",
            "source_file": "PW_CinePartition0.bin",
            "source_bytes_or_region": "full file size / 1296-byte pages",
            "supported_recordings": "10/10",
            "evidence_strength": "strong",
            "pipeline_utility": "duration and native page-time QC",
            "adoption_status": "adopt",
            "final_notebook_action": "preserve as supported native timing check",
            "allowed_claim": "fixed-page native PW timebase at working 500 Hz page rate",
            "forbidden_claim": "physiological waveform or clinical measurement",
            "reason": "all ten files divide exactly into 1296-byte pages and agree with AVI duration within about one to two video frames",
        },
        {
            "output_name": "20 Hz page-group timing counter",
            "source_file": "PW_CinePartition0.bin",
            "source_bytes_or_region": "bytes 1248..1249 per page, little-endian uint16-like",
            "supported_recordings": "10/10",
            "evidence_strength": "strong",
            "pipeline_utility": "acquisition integrity and page-order QC",
            "adoption_status": "adopt",
            "final_notebook_action": "preserve as native acquisition-integrity sidecar",
            "allowed_claim": "monotone native timing/counter-like field with median 25-page runs",
            "forbidden_claim": "physiological signal or Doppler measurement",
            "reason": "median run length is 25 pages in all recordings, implying 20 Hz at a 500 Hz page rate",
        },
        {
            "output_name": "hp_activity",
            "source_file": "PW_CinePartition0.bin",
            "source_bytes_or_region": "records 0..13, bytes [4:8], float32-like values",
            "supported_recordings": "selected recordings",
            "evidence_strength": "moderate",
            "pipeline_utility": "experimental arbitrary-unit timing/QC sidecar",
            "adoption_status": "adopt_minor",
            "final_notebook_action": "preserve with QC class and caveats",
            "allowed_claim": "experimental native-derived timing/QC activity signal",
            "forbidden_claim": "native velocity envelope, native Doppler index, pressure, or clinical waveform",
            "reason": "raw-byte reproduction succeeds for all recordings and HR agrees with candidate audio in many recordings, but robustness is recording-dependent",
        },
        {
            "output_name": "hp_activity_25",
            "source_file": "PW_CinePartition0.bin",
            "source_bytes_or_region": "records 0..13, bytes [4:8], shorter 25-page baseline",
            "supported_recordings": "sensitivity result only",
            "evidence_strength": "limited",
            "pipeline_utility": "filter-sensitivity check",
            "adoption_status": "candidate_future",
            "final_notebook_action": "keep as sensitivity/control result, not canonical output",
            "allowed_claim": "alternate high-pass activity variant for robustness checks",
            "forbidden_claim": "improved or validated physiological signal",
            "reason": "it was useful for filter-sensitivity testing but did not change the claim boundary",
        },
        {
            "output_name": "native beat times",
            "source_file": "PW_CinePartition0.bin",
            "source_bytes_or_region": "derived from hp_activity peaks",
            "supported_recordings": "not finally validated",
            "evidence_strength": "limited",
            "pipeline_utility": "possible internal QC only",
            "adoption_status": "candidate_future",
            "final_notebook_action": "do not export as final beat source in NB12",
            "allowed_claim": "candidate timing aid requiring validation",
            "forbidden_claim": "validated beat detector or clinical beat timing",
            "reason": "HR agreement exists in selected recordings, but beat timing was not validated as a final output",
        },
        {
            "output_name": "DcmRegionPara metadata",
            "source_file": "DcmRegionPara.txt",
            "source_bytes_or_region": "text metadata fields",
            "supported_recordings": "10/10",
            "evidence_strength": "strong for metadata presence",
            "pipeline_utility": "ROI, baseline, time-scale, and velocity-scale context for AVI workflow",
            "adoption_status": "adopt",
            "final_notebook_action": "preserve as metadata support",
            "allowed_claim": "metadata support for displayed AVI-based analysis",
            "forbidden_claim": "native waveform or clinical measurement by itself",
            "reason": "metadata exists across the batch and prior notebooks found practical calibration/context utility",
        },
        {
            "output_name": "FeParam metadata",
            "source_file": "VirtualMachine.bin",
            "source_bytes_or_region": "FeParam-like readable parameter blobs",
            "supported_recordings": "10/10 presence",
            "evidence_strength": "moderate for metadata-like extraction",
            "pipeline_utility": "candidate acquisition metadata support",
            "adoption_status": "adopt_minor",
            "final_notebook_action": "preserve as metadata-like support only",
            "allowed_claim": "candidate metadata-like parameter extraction",
            "forbidden_claim": "validated clinical meaning or waveform extraction",
            "reason": "NB08 supports readable parameter-like extraction but not clinical interpretation",
        },
        {
            "output_name": "BC raster-like buffer",
            "source_file": "BC_CinePartition1.bin",
            "source_bytes_or_region": "128-byte header plus uint8 raster-like payload",
            "supported_recordings": "10/10 presence, one exception layout",
            "evidence_strength": "moderate for raster-like internal buffer",
            "pipeline_utility": "possible future QC/context image-like sidecar",
            "adoption_status": "candidate_future",
            "final_notebook_action": "postpone deeper BC use",
            "allowed_claim": "structured native raster-like internal buffer",
            "forbidden_claim": "display-ready B-mode frame or anatomical image",
            "reason": "NB09 found structured raster-like data but not direct AVI frame recovery",
        },
        {
            "output_name": "high-rate 3500 Hz Doppler candidate",
            "source_file": "PW_CinePartition0.bin",
            "source_bytes_or_region": "records 0..13 as seven candidate I/Q pairs per page",
            "supported_recordings": "0/10 as useful high-rate Doppler evidence",
            "evidence_strength": "negative",
            "pipeline_utility": "none for current data",
            "adoption_status": "close_definitively",
            "final_notebook_action": "close for current data",
            "allowed_claim": "closed high-rate interpretation for current evidence",
            "forbidden_claim": "calibrated native Doppler velocity or high-rate complex baseband recovery",
            "reason": "phase rotation is near zero and magnitude HR does not reliably improve over the 500 Hz activity signal",
        },
        {
            "output_name": "native PW spectrogram",
            "source_file": "PW_CinePartition0.bin",
            "source_bytes_or_region": "searched PW dynamic/static regions",
            "supported_recordings": "0/10 validated",
            "evidence_strength": "negative",
            "pipeline_utility": "none for current data",
            "adoption_status": "close_definitively",
            "final_notebook_action": "close for current data",
            "allowed_claim": "native spectrogram recovery unsupported",
            "forbidden_claim": "recovered display PW spectrogram",
            "reason": "NB10 did not recover raw or display-ready spectrogram raster",
        },
        {
            "output_name": "native velocity envelope",
            "source_file": "PW_CinePartition0.bin",
            "source_bytes_or_region": "not validated",
            "supported_recordings": "0/10 validated",
            "evidence_strength": "negative",
            "pipeline_utility": "none for current data",
            "adoption_status": "close_definitively",
            "final_notebook_action": "close for current data",
            "allowed_claim": "unsupported by current native-file evidence",
            "forbidden_claim": "native PSV, EDV, RI, PI, VTI, or calibrated velocity",
            "reason": "no calibrated native velocity envelope was reproduced or validated",
        },
        {
            "output_name": "PHYSIO trace",
            "source_file": "native export metadata/config",
            "source_bytes_or_region": "declared or implied metadata, not persisted trace",
            "supported_recordings": "0/10 persisted trace validated",
            "evidence_strength": "negative",
            "pipeline_utility": "none for current data",
            "adoption_status": "close_definitively",
            "final_notebook_action": "close for current data",
            "allowed_claim": "PHYSIO trace recovery unsupported",
            "forbidden_claim": "persisted physiological waveform recovery",
            "reason": "no duration-scaled persisted PHYSIO partition was validated",
        },
        {
            "output_name": "pressure or clinical metrics",
            "source_file": "not supported",
            "source_bytes_or_region": "not supported",
            "supported_recordings": "0/10",
            "evidence_strength": "negative",
            "pipeline_utility": "none",
            "adoption_status": "close_definitively",
            "final_notebook_action": "explicitly forbid",
            "allowed_claim": "not supported",
            "forbidden_claim": "pressure, mmHg, clinical-grade measurement, diagnostic interpretation",
            "reason": "no current project evidence validates pressure or clinical measurements from native files",
        },
    ]

    return pd.DataFrame(rows)


In [29]:
native_file_utility_matrix = build_native_file_utility_matrix()

print("Native file utility matrix")
print(f"Rows: {len(native_file_utility_matrix)}")
print("Adoption status counts:")
display(native_file_utility_matrix["adoption_status"].value_counts().rename_axis("adoption_status").reset_index(name="n"))

display(native_file_utility_matrix)

Native file utility matrix
Rows: 13
Adoption status counts:


,adoption_status,n
0,close_definitively,5
1,adopt,3
2,candidate_future,3
3,adopt_minor,2


,output_name,source_file,source_bytes_or_region,supported_recordings,evidence_strength,pipeline_utility,adoption_status,final_notebook_action,allowed_claim,forbidden_claim,reason
0,PW page timebase,PW_CinePartition0.bin,full file size / 1296-byte pages,10/10,strong,duration and native page-time QC,adopt,preserve as supported native timing check,fixed-page native PW timebase at working 500 H...,physiological waveform or clinical measurement,all ten files divide exactly into 1296-byte pa...
1,20 Hz page-group timing counter,PW_CinePartition0.bin,"bytes 1248..1249 per page, little-endian uint1...",10/10,strong,acquisition integrity and page-order QC,adopt,preserve as native acquisition-integrity sidecar,monotone native timing/counter-like field with...,physiological signal or Doppler measurement,median run length is 25 pages in all recording...
2,hp_activity,PW_CinePartition0.bin,"records 0..13, bytes [4:8], float32-like values",selected recordings,moderate,experimental arbitrary-unit timing/QC sidecar,adopt_minor,preserve with QC class and caveats,experimental native-derived timing/QC activity...,"native velocity envelope, native Doppler index...",raw-byte reproduction succeeds for all recordi...
3,hp_activity_25,PW_CinePartition0.bin,"records 0..13, bytes [4:8], shorter 25-page ba...",sensitivity result only,limited,filter-sensitivity check,candidate_future,"keep as sensitivity/control result, not canoni...",alternate high-pass activity variant for robus...,improved or validated physiological signal,it was useful for filter-sensitivity testing b...
4,native beat times,PW_CinePartition0.bin,derived from hp_activity peaks,not finally validated,limited,possible internal QC only,candidate_future,do not export as final beat source in NB12,candidate timing aid requiring validation,validated beat detector or clinical beat timing,"HR agreement exists in selected recordings, bu..."
5,DcmRegionPara metadata,DcmRegionPara.txt,text metadata fields,10/10,strong for metadata presence,"ROI, baseline, time-scale, and velocity-scale ...",adopt,preserve as metadata support,metadata support for displayed AVI-based analysis,native waveform or clinical measurement by itself,metadata exists across the batch and prior not...
6,FeParam metadata,VirtualMachine.bin,FeParam-like readable parameter blobs,10/10 presence,moderate for metadata-like extraction,candidate acquisition metadata support,adopt_minor,preserve as metadata-like support only,candidate metadata-like parameter extraction,validated clinical meaning or waveform extraction,NB08 supports readable parameter-like extracti...
7,BC raster-like buffer,BC_CinePartition1.bin,128-byte header plus uint8 raster-like payload,"10/10 presence, one exception layout",moderate for raster-like internal buffer,possible future QC/context image-like sidecar,candidate_future,postpone deeper BC use,structured native raster-like internal buffer,display-ready B-mode frame or anatomical image,NB09 found structured raster-like data but not...
8,high-rate 3500 Hz Doppler candidate,PW_CinePartition0.bin,records 0..13 as seven candidate I/Q pairs per...,0/10 as useful high-rate Doppler evidence,negative,none for current data,close_definitively,close for current data,closed high-rate interpretation for current ev...,calibrated native Doppler velocity or high-rat...,phase rotation is near zero and magnitude HR d...
9,native PW spectrogram,PW_CinePartition0.bin,searched PW dynamic/static regions,0/10 validated,negative,none for current data,close_definitively,close for current data,native spectrogram recovery unsupported,recovered display PW spectrogram,NB10 did not recover raw or display-ready spec...


### Native file utility matrix interpretation

Result classification:

This section supports a final NB12 utility split between native outputs to keep, native outputs to treat as candidates, and native extraction paths to close for the current data.

Evidence-supported observations:

- Thirteen native-file outputs or extraction paths were summarized.
- Three outputs were marked `adopt`: PW page timebase, 20 Hz page-group timing counter, and `DcmRegionPara` metadata.
- Two outputs were marked `adopt_minor`: `hp_activity` and FeParam metadata.
- Three outputs were marked `candidate_future`: `hp_activity_25`, native beat times, and the BC raster-like buffer.
- Five paths were marked `close_definitively` for the current data: high-rate `3500 Hz` Doppler, native PW spectrogram, native velocity envelope, PHYSIO trace, and pressure or clinical metrics.

What this supports:

- Native files have practical value as metadata, timebase, acquisition-integrity, and experimental timing/QC support.
- `hp_activity` should be preserved only with recording-level QC caveats.
- Unsupported extraction paths can be closed for the current project phase.
- The practical Doppler feature pipeline should remain AVI/image/audio based, with native files used as sidecar support.

What this does not support:

- This matrix does not validate calibrated native velocity.
- This matrix does not validate native PSV, EDV, RI, PI, VTI, pressure, or clinical interpretation.
- This matrix does not validate native spectrogram recovery or PHYSIO trace recovery.
- `close_definitively` means closed for the current data and current project evidence, not impossible in all future vendor-format contexts.


## Final native claim boundary table

### What this tests

This section converts the utility matrix into a compact final claim-boundary table.

### Why this matters

NB12 needs a clear final boundary between supported native-file claims, candidate future claims, and unsupported or closed claims. This table is intended to make the final checkpoint summary easier to audit.


In [30]:
def build_final_native_claim_boundary(utility_matrix, signal_classification):
    """This function builds a compact final NB12 claim-boundary table."""
    class_counts = (
        signal_classification["nb12_evidence_class"]
        .value_counts()
        .to_dict()
    )

    native_supported_count = (
        class_counts.get("native_specific_supported", 0)
        + class_counts.get("native_specific_with_qc_caveat", 0)
        + class_counts.get("borderline_native_specific_with_qc_caveat", 0)
        + class_counts.get("borderline_native_specific_supported", 0)
    )

    reject_count = (
        class_counts.get("reject_supported", 0)
        + class_counts.get("reject_with_audio_agreement_caveat", 0)
    )

    rows = [
        {
            "claim_area": "PW page structure",
            "final_status": "supported",
            "evidence_summary": "All ten PW files divide exactly into 1296-byte pages.",
            "allowed_claim": "fixed-page native PW stream with working 500 Hz page timebase",
            "not_allowed_claim": "physiological waveform or clinical measurement",
            "final_action": "keep as native timing/QC foundation",
        },
        {
            "claim_area": "Native duration check",
            "final_status": "supported",
            "evidence_summary": "Native duration from n_pages / 500 agrees with linked AVI duration within about one to two 30 fps frames.",
            "allowed_claim": "recording-level native duration check",
            "not_allowed_claim": "sample-perfect clinical timing reference",
            "final_action": "keep as recording-level QC",
        },
        {
            "claim_area": "20 Hz page-group counter",
            "final_status": "supported",
            "evidence_summary": "All ten recordings show median 25-page runs at a 500 Hz page rate.",
            "allowed_claim": "monotone acquisition-integrity counter-like field",
            "not_allowed_claim": "physiological or Doppler signal",
            "final_action": "keep as page-order and acquisition-integrity QC",
        },
        {
            "claim_area": "hp_activity timing/QC",
            "final_status": "supported with caveats",
            "evidence_summary": f"Raw-byte reproduction succeeds in 10/10 recordings; {native_supported_count} recordings support native-specific or borderline timing/QC interpretation.",
            "allowed_claim": "experimental arbitrary-unit native timing/QC sidecar",
            "not_allowed_claim": "native velocity envelope, Doppler index, pressure, or clinical waveform",
            "final_action": "keep as minor sidecar with recording-level QC labels",
        },
        {
            "claim_area": "Reject examples",
            "final_status": "supported",
            "evidence_summary": f"{reject_count} recordings support reject classification by instability, harmonic ambiguity, or native-audio disagreement.",
            "allowed_claim": "recordings can fail native timing/QC eligibility",
            "not_allowed_claim": "all native files contain usable cardiac timing",
            "final_action": "preserve reject labels as QC evidence",
        },
        {
            "claim_area": "DcmRegionPara metadata",
            "final_status": "supported",
            "evidence_summary": "Metadata exists across all ten recordings and supports displayed AVI workflow context.",
            "allowed_claim": "metadata support for ROI, baseline, time-scale, and velocity-scale context",
            "not_allowed_claim": "native waveform extraction or clinical measurement by itself",
            "final_action": "keep as metadata support",
        },
        {
            "claim_area": "FeParam metadata",
            "final_status": "candidate support",
            "evidence_summary": "VirtualMachine.bin exists across all ten recordings and prior work found readable parameter-like blobs.",
            "allowed_claim": "candidate metadata-like extraction",
            "not_allowed_claim": "validated clinical meaning or waveform source",
            "final_action": "keep as minor metadata support",
        },
        {
            "claim_area": "BC raster-like buffer",
            "final_status": "candidate future",
            "evidence_summary": "BC files exist across all ten recordings, with one known exception layout.",
            "allowed_claim": "structured raster-like native internal buffer",
            "not_allowed_claim": "display-ready B-mode frame or validated anatomical image",
            "final_action": "postpone deeper BC use",
        },
        {
            "claim_area": "High-rate 3500 Hz Doppler",
            "final_status": "closed for current data",
            "evidence_summary": "Candidate seven-pair stream has near-zero net page phase and does not reliably improve HR agreement.",
            "allowed_claim": "high-rate interpretation is unsupported for current evidence",
            "not_allowed_claim": "calibrated high-rate native complex Doppler or native velocity recovery",
            "final_action": "close for current project phase",
        },
        {
            "claim_area": "Native spectrogram recovery",
            "final_status": "closed for current data",
            "evidence_summary": "No raw or display-ready PW spectrogram raster was validated.",
            "allowed_claim": "native PW spectrogram recovery remains unsupported",
            "not_allowed_claim": "recovered native display spectrogram",
            "final_action": "close for current project phase",
        },
        {
            "claim_area": "Native velocity and Doppler indices",
            "final_status": "closed for current data",
            "evidence_summary": "No calibrated native velocity envelope or Doppler index source was validated.",
            "allowed_claim": "unsupported by current native-file evidence",
            "not_allowed_claim": "native PSV, EDV, RI, PI, VTI, or calibrated velocity",
            "final_action": "explicitly forbid",
        },
        {
            "claim_area": "PHYSIO trace",
            "final_status": "closed for current data",
            "evidence_summary": "No duration-scaled persisted PHYSIO trace was validated.",
            "allowed_claim": "PHYSIO recovery unsupported",
            "not_allowed_claim": "persisted physiological waveform recovery",
            "final_action": "close for current project phase",
        },
        {
            "claim_area": "Pressure and clinical metrics",
            "final_status": "not supported",
            "evidence_summary": "No current evidence supports pressure, mmHg, diagnostic output, or clinical-grade measurements from native files.",
            "allowed_claim": "not supported",
            "not_allowed_claim": "pressure, mmHg, clinical-grade measurement, diagnostic interpretation",
            "final_action": "explicitly forbid",
        },
    ]

    claim_boundary = pd.DataFrame(rows)

    status_order = {
        "supported": 0,
        "supported with caveats": 1,
        "candidate support": 2,
        "candidate future": 3,
        "closed for current data": 4,
        "not supported": 5,
    }

    claim_boundary["status_order"] = claim_boundary["final_status"].map(status_order)
    claim_boundary = claim_boundary.sort_values(["status_order", "claim_area"]).drop(columns="status_order")
    claim_boundary = claim_boundary.reset_index(drop=True)

    return claim_boundary

In [31]:
final_native_claim_boundary = build_final_native_claim_boundary(
    native_file_utility_matrix,
    nb12_native_signal_classification,
)

print("Final native claim-boundary table")
print(f"Rows: {len(final_native_claim_boundary)}")
print("Final status counts:")
display(final_native_claim_boundary["final_status"].value_counts().rename_axis("final_status").reset_index(name="n"))

display(final_native_claim_boundary)

Final native claim-boundary table
Rows: 13
Final status counts:


,final_status,n
0,supported,5
1,closed for current data,4
2,supported with caveats,1
3,candidate support,1
4,candidate future,1
5,not supported,1


,claim_area,final_status,evidence_summary,allowed_claim,not_allowed_claim,final_action
0,20 Hz page-group counter,supported,All ten recordings show median 25-page runs at...,monotone acquisition-integrity counter-like field,physiological or Doppler signal,keep as page-order and acquisition-integrity QC
1,DcmRegionPara metadata,supported,Metadata exists across all ten recordings and ...,"metadata support for ROI, baseline, time-scale...",native waveform extraction or clinical measure...,keep as metadata support
2,Native duration check,supported,Native duration from n_pages / 500 agrees with...,recording-level native duration check,sample-perfect clinical timing reference,keep as recording-level QC
3,PW page structure,supported,All ten PW files divide exactly into 1296-byte...,fixed-page native PW stream with working 500 H...,physiological waveform or clinical measurement,keep as native timing/QC foundation
4,Reject examples,supported,2 recordings support reject classification by ...,recordings can fail native timing/QC eligibility,all native files contain usable cardiac timing,preserve reject labels as QC evidence
5,hp_activity timing/QC,supported with caveats,Raw-byte reproduction succeeds in 10/10 record...,experimental arbitrary-unit native timing/QC s...,"native velocity envelope, Doppler index, press...",keep as minor sidecar with recording-level QC ...
6,FeParam metadata,candidate support,VirtualMachine.bin exists across all ten recor...,candidate metadata-like extraction,validated clinical meaning or waveform source,keep as minor metadata support
7,BC raster-like buffer,candidate future,"BC files exist across all ten recordings, with...",structured raster-like native internal buffer,display-ready B-mode frame or validated anatom...,postpone deeper BC use
8,High-rate 3500 Hz Doppler,closed for current data,Candidate seven-pair stream has near-zero net ...,high-rate interpretation is unsupported for cu...,calibrated high-rate native complex Doppler or...,close for current project phase
9,Native spectrogram recovery,closed for current data,No raw or display-ready PW spectrogram raster ...,native PW spectrogram recovery remains unsuppo...,recovered native display spectrogram,close for current project phase


### Final native claim-boundary table interpretation

Result classification:

This section supports the final NB12 claim boundary for native Mindray file exploration in the current project phase.

Evidence-supported observations:

- Thirteen claim areas were summarized.
- Five claim areas were classified as `supported`: PW page structure, native duration check, 20 Hz page-group counter, DcmRegionPara metadata, and reject-example eligibility.
- One claim area, `hp_activity timing/QC`, was classified as `supported with caveats`.
- FeParam metadata was classified as `candidate support`.
- The BC raster-like buffer was classified as `candidate future`.
- Four claim areas were closed for the current data: high-rate `3500 Hz` Doppler, native spectrogram recovery, native velocity and Doppler indices, and PHYSIO trace.
- Pressure and clinical metrics were classified as `not supported`.

What this supports:

- Native files should be kept as a metadata, timebase, acquisition-integrity, and experimental timing/QC support layer.
- The surviving native signal claim should stay narrow: `hp_activity` is an arbitrary-unit timing/QC sidecar with recording-level caveats.
- Unsupported native extraction directions can be closed for this project phase.
- The practical Doppler feature pipeline should remain based on linked AVI image/audio, with native files used as sidecar support.

What this does not support:

- This table does not validate native velocity, native Doppler indices, pressure, PHYSIO recovery, native spectrogram recovery, or clinical interpretation.
- This table does not validate native beat timing as a final export.
- `Closed for current data` does not mean impossible in every future vendor-format context; it means unsupported by the current project evidence.


## Final checkpoint readiness check

### What this tests

This section checks whether the core NB12 checkpoint tables exist, have expected row counts, and have clear intended output filenames.

### Why this matters

NB12 should save only useful final checkpoint artifacts. Before creating folders or writing files, this section previews which tables are ready and what each table is intended to document.


In [32]:
def build_checkpoint_readiness_table():
    """This function summarizes NB12 checkpoint tables without writing files."""
    checkpoint_specs = [
        {
            "object_name": "native_manifest",
            "intended_filename": "nb12_native_recording_manifest.csv",
            "purpose": "recording-level native file presence and candidate AVI linkage",
            "expected_rows": 10,
        },
        {
            "object_name": "pw_timebase_qc",
            "intended_filename": "nb12_native_timebase_qc.csv",
            "purpose": "1296-byte page decomposition and native duration comparison",
            "expected_rows": 10,
        },
        {
            "object_name": "pw_counter_qc",
            "intended_filename": "nb12_native_counter_qc.csv",
            "purpose": "20 Hz page-group counter and acquisition-integrity summary",
            "expected_rows": 10,
        },
        {
            "object_name": "hp_activity_summary",
            "intended_filename": "nb12_hp_activity_reproduction_summary.csv",
            "purpose": "raw-byte hp_activity reproduction summary",
            "expected_rows": 10,
        },
        {
            "object_name": "native_hr_summary",
            "intended_filename": "nb12_native_hr_window_summary.csv",
            "purpose": "native HR and window stability summary from hp_activity",
            "expected_rows": 10,
        },
        {
            "object_name": "native_hr_robustness",
            "intended_filename": "nb12_native_hr_robustness_qc.csv",
            "purpose": "subset disagreement and filter sensitivity QC",
            "expected_rows": 10,
        },
        {
            "object_name": "nb12_audio_comparison",
            "intended_filename": "nb12_candidate_audio_reference_comparison.csv",
            "purpose": "NB12 native HR compared with candidate audio HR",
            "expected_rows": 10,
        },
        {
            "object_name": "nb12_native_signal_classification",
            "intended_filename": "nb12_native_signal_classification.csv",
            "purpose": "recording-level native timing/QC classification synthesis",
            "expected_rows": 10,
        },
        {
            "object_name": "high_rate_closure_summary",
            "intended_filename": "nb12_high_rate_doppler_closure.csv",
            "purpose": "3500 Hz high-rate Doppler closure evidence",
            "expected_rows": 10,
        },
        {
            "object_name": "native_file_utility_matrix",
            "intended_filename": "nb12_native_file_utility_matrix.csv",
            "purpose": "final utility and adoption matrix",
            "expected_rows": 13,
        },
        {
            "object_name": "final_native_claim_boundary",
            "intended_filename": "nb12_final_native_claim_boundary.csv",
            "purpose": "final supported/candidate/closed claim boundary",
            "expected_rows": 13,
        },
    ]

    rows = []

    for spec in checkpoint_specs:
        object_name = spec["object_name"]
        table = globals().get(object_name)

        exists = isinstance(table, pd.DataFrame)
        n_rows = len(table) if exists else np.nan
        n_columns = len(table.columns) if exists else np.nan

        rows.append(
            {
                **spec,
                "object_exists": exists,
                "n_rows": n_rows,
                "n_columns": n_columns,
                "row_count_matches_expected": bool(exists and n_rows == spec["expected_rows"]),
                "ready_to_save": bool(exists and n_rows > 0),
            }
        )

    return pd.DataFrame(rows)

In [33]:
checkpoint_readiness = build_checkpoint_readiness_table()

print("NB12 checkpoint readiness")
print(f"Tables listed: {len(checkpoint_readiness)}")
print(f"Tables existing: {int(checkpoint_readiness['object_exists'].sum())} / {len(checkpoint_readiness)}")
print(f"Tables ready to save: {int(checkpoint_readiness['ready_to_save'].sum())} / {len(checkpoint_readiness)}")
print(f"SAVE_OUTPUTS is currently: {SAVE_OUTPUTS}")

display(checkpoint_readiness)

NB12 checkpoint readiness
Tables listed: 11
Tables existing: 11 / 11
Tables ready to save: 11 / 11
SAVE_OUTPUTS is currently: False


,object_name,intended_filename,purpose,expected_rows,object_exists,n_rows,n_columns,row_count_matches_expected,ready_to_save
0,native_manifest,nb12_native_recording_manifest.csv,recording-level native file presence and candi...,10,True,10,18,True,True
1,pw_timebase_qc,nb12_native_timebase_qc.csv,1296-byte page decomposition and native durati...,10,True,10,14,True,True
2,pw_counter_qc,nb12_native_counter_qc.csv,20 Hz page-group counter and acquisition-integ...,10,True,10,16,True,True
3,hp_activity_summary,nb12_hp_activity_reproduction_summary.csv,raw-byte hp_activity reproduction summary,10,True,10,14,True,True
4,native_hr_summary,nb12_native_hr_window_summary.csv,native HR and window stability summary from hp...,10,True,10,9,True,True
5,native_hr_robustness,nb12_native_hr_robustness_qc.csv,subset disagreement and filter sensitivity QC,10,True,10,13,True,True
6,nb12_audio_comparison,nb12_candidate_audio_reference_comparison.csv,NB12 native HR compared with candidate audio HR,10,True,10,13,True,True
7,nb12_native_signal_classification,nb12_native_signal_classification.csv,recording-level native timing/QC classificatio...,10,True,10,13,True,True
8,high_rate_closure_summary,nb12_high_rate_doppler_closure.csv,3500 Hz high-rate Doppler closure evidence,10,True,10,20,True,True
9,native_file_utility_matrix,nb12_native_file_utility_matrix.csv,final utility and adoption matrix,13,True,13,11,True,True


### Final checkpoint readiness interpretation

Result classification:

This section supports NB12 checkpoint readiness.

Evidence-supported observations:

- Eleven checkpoint tables were listed.
- All eleven checkpoint tables exist in notebook memory.
- All eleven checkpoint tables are ready to save.
- All row counts match the expected counts.
- `SAVE_OUTPUTS` is currently `False`, so no checkpoint files have been written by this readiness check.

What this supports:

- NB12 has produced the core tables needed for final native-file closure.
- The notebook can proceed to an optional save step after review.
- The checkpoint artifacts have clear intended filenames and purposes.

What this does not support:

- This readiness check does not write files.
- This readiness check does not add new scientific evidence.
- This readiness check does not validate native velocity, pressure, native Doppler indices, spectrogram recovery, or clinical interpretation.


## Optional checkpoint artifact save

### What this tests

This section optionally writes reviewed NB12 checkpoint tables to a dedicated report folder.

### Why this matters

NB12 should save only useful final checkpoint artifacts. The previous readiness check confirmed that the intended tables exist and have expected row counts.


In [34]:
checkpoint_tables_to_save = {
    "nb12_native_recording_manifest.csv": native_manifest,
    "nb12_native_timebase_qc.csv": pw_timebase_qc,
    "nb12_native_counter_qc.csv": pw_counter_qc,
    "nb12_hp_activity_reproduction_summary.csv": hp_activity_summary,
    "nb12_native_hr_window_summary.csv": native_hr_summary,
    "nb12_native_hr_robustness_qc.csv": native_hr_robustness,
    "nb12_candidate_audio_reference_comparison.csv": nb12_audio_comparison,
    "nb12_native_signal_classification.csv": nb12_native_signal_classification,
    "nb12_high_rate_doppler_closure.csv": high_rate_closure_summary,
    "nb12_native_file_utility_matrix.csv": native_file_utility_matrix,
    "nb12_final_native_claim_boundary.csv": final_native_claim_boundary,
}

save_plan_rows = []

for filename, table in checkpoint_tables_to_save.items():
    output_path = REPORT_ROOT / filename

    save_plan_rows.append(
        {
            "filename": filename,
            "output_path": output_path,
            "n_rows": len(table),
            "n_columns": len(table.columns),
            "will_write": bool(SAVE_OUTPUTS),
        }
    )

save_plan = pd.DataFrame(save_plan_rows)

if SAVE_OUTPUTS:
    REPORT_ROOT.mkdir(parents=True, exist_ok=True)

    for filename, table in checkpoint_tables_to_save.items():
        output_path = REPORT_ROOT / filename
        table.to_csv(output_path, index=False)

    print(f"Saved {len(checkpoint_tables_to_save)} NB12 checkpoint tables to:")
    print(REPORT_ROOT)
else:
    print("SAVE_OUTPUTS is False. No files were written.")
    print("To save reviewed checkpoint tables, set SAVE_OUTPUTS = True and rerun this cell.")

display(save_plan)

SAVE_OUTPUTS is False. No files were written.
To save reviewed checkpoint tables, set SAVE_OUTPUTS = True and rerun this cell.


,filename,output_path,n_rows,n_columns,will_write
0,nb12_native_recording_manifest.csv,D:\code\DopplerLab\reports\nb12_native_file_cl...,10,18,False
1,nb12_native_timebase_qc.csv,D:\code\DopplerLab\reports\nb12_native_file_cl...,10,14,False
2,nb12_native_counter_qc.csv,D:\code\DopplerLab\reports\nb12_native_file_cl...,10,16,False
3,nb12_hp_activity_reproduction_summary.csv,D:\code\DopplerLab\reports\nb12_native_file_cl...,10,14,False
4,nb12_native_hr_window_summary.csv,D:\code\DopplerLab\reports\nb12_native_file_cl...,10,9,False
5,nb12_native_hr_robustness_qc.csv,D:\code\DopplerLab\reports\nb12_native_file_cl...,10,13,False
6,nb12_candidate_audio_reference_comparison.csv,D:\code\DopplerLab\reports\nb12_native_file_cl...,10,13,False
7,nb12_native_signal_classification.csv,D:\code\DopplerLab\reports\nb12_native_file_cl...,10,13,False
8,nb12_high_rate_doppler_closure.csv,D:\code\DopplerLab\reports\nb12_native_file_cl...,10,20,False
9,nb12_native_file_utility_matrix.csv,D:\code\DopplerLab\reports\nb12_native_file_cl...,13,11,False


### Optional checkpoint artifact save interpretation

Result classification:

This section previews the final NB12 checkpoint artifact save plan.

Evidence-supported observations:

- Eleven reviewed checkpoint tables are included in the save plan.
- Each table has the expected row count from the readiness check.
- All planned outputs target `reports/nb12_native_file_closure/`.
- `SAVE_OUTPUTS` is currently `False`, so no files were written.

What this supports:

- The notebook has a controlled save plan for final checkpoint artifacts.
- The save cell can be rerun with `SAVE_OUTPUTS = True` if checkpoint CSV files should be persisted.
- The current run did not modify repository files.

What this does not support:

- This section does not add new native-file evidence.
- This section does not validate native velocity, pressure, native Doppler indices, spectrogram recovery, PHYSIO recovery, or clinical interpretation.


## Final checkpoint - Native file exploration closure

### Result classification

This notebook supports a narrow native-file utility claim.

Native Mindray files provide useful metadata, timebase, acquisition-integrity, and experimental timing/QC support. They do not provide a validated standalone Doppler waveform or clinical measurement source.

### What is supported

- `PW_CinePartition0.bin` behaves as a fixed-page native PW stream in the current ten-recording batch.
- The supported page size is `1296 bytes`.
- The working page rate is `500 Hz`.
- Native duration from `n_pages / 500` is a useful recording-level timebase check.
- Bytes `1248..1249` provide a monotone page-group timing/counter-like field.
- The page-group field has median `25` page runs, consistent with a `20 Hz` update rate at a `500 Hz` page rate.
- Records `0..13`, bytes `[4:8]`, support reproducible float32-like dynamic values.
- `hp_activity` can be reproduced directly from raw bytes and preserved as an experimental arbitrary-unit timing/QC sidecar.
- Native timing/QC usefulness is recording-dependent and requires QC labels.
- `DcmRegionPara` metadata supports ROI, baseline, time-scale, and velocity-scale context for the displayed AVI-based workflow.
- FeParam-like extraction from `VirtualMachine.bin` remains useful as candidate metadata support.

### Recording-level native timing/QC interpretation

The current NB12 synthesis supports the following cautious grouping:

- `native_specific_supported`: `202606130417060004SMP`, `202606130422440006SMP`
- `native_specific_with_qc_caveat`: `202606130411060002SMP`, `202606130413540003SMP`, `202606130420260005SMP`
- `borderline_native_specific_with_qc_caveat`: `202606130437480012SMP`
- `audio_only_candidate_supported`: `202606130434130010SMP`
- `audio_only_candidate_with_native_agreement_conflict`: `202606130433280009SMP`
- `reject_supported`: `202606130426500007SMP`, `202606130430380008SMP`

These labels describe experimental native timing/QC eligibility only.

They do not describe clinical quality, diagnostic status, or validated Doppler measurements.

### What is useful for the future pipeline

- Native page timebase can check duration agreement.
- The 20 Hz page-group timing/counter-like field can check page order and acquisition integrity.
- `hp_activity` can provide a native timing/QC sidecar in selected recordings.
- Native QC classes can help label native-specific, audio-only candidate, borderline, and reject recordings.
- `DcmRegionPara` metadata can reduce manual setup for the AVI/image/audio pipeline.
- FeParam-like metadata can remain a candidate support layer, with cautious interpretation.

### What remains candidate or postponed

- `hp_activity_25` is useful as a filter-sensitivity check, not as the canonical signal.
- Native beat times should remain candidate internal QC only unless separately validated.
- `BC_CinePartition1.bin` contains a structured raster-like native buffer, but it is not a validated display-ready B-mode frame. Deeper BC use should be postponed.
- FeParam-like outputs should remain metadata-like support unless their meanings are independently validated.

### What is closed for the current data

- High-rate `3500 Hz` Doppler interpretation is closed for the current data.
- Native PW spectrogram recovery is closed for the current data.
- Native velocity envelope recovery is closed for the current data.
- Native PSV, EDV, RI, PI, and VTI extraction is unsupported.
- PHYSIO trace recovery is unsupported.
- Pressure, mmHg, clinical-grade measurement, and diagnostic interpretation are unsupported.

